In [ ]:
/**
 * From Local Development to Production:
 * > Until now, everything runs on localhost and this is the phase where things get to
 *   start serious. The "Money" part gets involved here, and this is where your system
 *   design can decide if you are going to have a Pizza party this weekend or not.
 * > Production Infrastructure:
 *   - Cloud Infrastructure - AWS, Digital Ocean, etc
 *   - CDNs (Content Delivery Network) -  bringing data closer to users
 *   - Load balancers and traffic distribution
 *   - Understanding meaning behind CPU and memory
 *   - Cost related to CPUs and Memory
 *   - Virtul Private Clouds (VPCs)
 *   - Public IP and Private IP Addressing
 *   - Bad Boys do DDoS
 *   - When to Scale Out and Scale In?
*/

In [ ]:
/**
 * Cloud Native System Design [AWS]
 * > In this phase, we will be discussing core AWS Services which help you scale
 *   your system with managed services.
 * > AWS Services:
 *   - Core AWS Services
 *   - AWS EC2, ECS and Lambda: Compute at scale
 *   - RDS, DynamoDB and Aurora: Managed database options
 *   - API Gateway + Load Balancers: Managing traffic in the cloud
 *   - S3 and CloudFront: Storage + global content delivery
 *   - Monitoring and Observability using CloudWatch
 */

In [ ]:
/**
 * Cloud Infrastructure (AWS):
 * > AWS, Linode, GCP, Azure, etc. are cloud providers, meaning they
 *   have large data centers at multiple locations around the world.
 *   These locations are called "Regions", and within each Region there
 *   are multiple "Availability Zones (AZs)" — physically separate data
 *   centers with independent power, cooling, and networking.
 *
 * > To understand how a data center is built:
 *   - They have a large plot of land.
 *   - On that plot, large buildings are constructed.
 *   - Inside those buildings, there are multiple dedicated sections:
 *     - One section only for hard disks / storage
 *     - One section only for cooling systems
 *     - Sections for servers (CPUs, RAM, etc.)
 *     - Sections for networking hardware
 *   - They also have external components like:
 *     - Network Cards / routers that connect the data center to the internet
 *       via high-speed fiber optic links
 *   - Together, all of this forms a complete data center — this is
 *     called the Physical Layer.
 *
 *   Physical Layer Overview:
 *
 *     +-------------------------------------------------------------+
 *     |                        PLOT / CAMPUS                        |
 *     |  +-------------------------------------------------------+  |
 *     |  |                      BUILDING                         |  |
 *     |  |  +-----------+  +----------+  +-------+  +--------+   |  |
 *     |  |  |  Storage  |  | Servers  |  | Power |  | Cooling|   |  |
 *     |  |  |(Hard Disk)|  |(CPU, RAM)|  | (UPS) |  | Systems|   |  |
 *     |  |  +-----------+  +----------+  +-------+  +--------+   |  |
 *     |  +-------------------------------------------------------+  |
 *     |  +-------------------+  +--------------------------------+  |
 *     |  |   Network Cards   |  |         Hypervisor             |  |
 *     |  | (Internet Access) |  | (Virtualizes Physical Layer)   |  |
 *     |  +-------------------+  +--------------------------------+  |
 *     +-------------------------------------------------------------+
 *
 *   - Direct access to the Physical Layer cannot be given to every user,
 *     so on top of it, AWS has built a Hypervisor. The Hypervisor is
 *     responsible for virtualizing the physical machine into multiple
 *     Virtual Machines (VMs), allowing many users to share the same
 *     physical hardware in isolation.
 *     (AWS uses a custom hypervisor called the "AWS Nitro System".)
 *
 *   - To interact with the Hypervisor and request resources, AWS has
 *     built APIs (accessible via CLI, SDKs, or REST) on top of it:
 *
 *                                          [User]
 *                                             |
 *                               +-------------+-------------+
 *                               |       AWS APIs / CLI      |
 *                               |  /create-machine          |
 *                               |  /delete-machine          |
 *                               |  /create-loadbalancer     |
 *                               |  /attach-loadbalancer     |
 *                               +-------------+-------------+
 *                                             |
 *                                             V
 *     +-------------------------------------------------------------+
 *     |  +-------------------+  +--------------------------------+  |
 *     |  |   Network Cards   |  |         Hypervisor             |  |
 *     |  | (Internet Access) |  |    (AWS Nitro System)          |  |
 *     |  +-------------------+  +--------------------------------+  |
 *     |  +-------------------------------------------------------+  |
 *     |  |                   Physical Layer                      |  |
 *     |  |  +-----------+  +----------+  +-------+  +--------+   |  |
 *     |  |  |  Storage  |  | Servers  |  | Power |  | Cooling|   |  |
 *     |  |  +-----------+  +----------+  +-------+  +--------+   |  |
 *     |  +-------------------------------------------------------+  |
 *     +-------------------------------------------------------------+
 *
 *   - The user interacts with AWS through these APIs, which communicate
 *     with the Hypervisor. The Hypervisor has access to the Physical Layer
 *     and spins up (provisions) the requested resources. The user is then
 *     charged based on usage (pay-as-you-go model).
 *
 *   - In the early days, AWS only provided S3 (Simple Storage Service),
 *     which was accessible only via CLI. Every single operation — creating
 *     buckets, uploading files, or even fetching bills — required writing
 *     manual commands.
 */

In [ ]:
/**
 * AWS Management Console:
 *   - To solve the problem of CLI-only access, AWS built the AWS Management
 *     Console — a web-based graphical UI that sits on top of the exact same
 *     underlying APIs. It makes the same API calls on behalf of the user,
 *     but through a visual interface instead of a CLI.
 *
 *   - The AWS Management Console helps us visualize and manage all AWS
 *     resources in one place — EC2 instances, S3 buckets, billing, logs,
 *     IAM users, and much more.
 *
 *   How different interfaces talk to AWS:
 *   a. AWS Management Console (Web UI / GUI)
 *      - Click-based interface
 *      - Visualize resources, billing, logs, etc.
 *   b. AWS CLI
 *      - Command-line access
 *      - Useful for scripting and automation
 *   c. AWS SDKs (JavaScript, Python, Java, etc.)
 *      - Programmatic access from application code
 *
 *     +---------------------------------------------------------------+
 *     |                           [User]                              |
 *     |                              |                                |
 *     |          +-------------------+-------------------+            |
 *     |          |                   |                   |            |
 *     |          V                   V                   V            |
 *     |    +----------+      +-------------+     +-------------+      |
 *     |    |  AWS     |      |   AWS CLI   |     |  AWS SDKs   |      |
 *     |    |Management|      | (Terminal / |     | (JS, Python,|      |
 *     |    | Console  |      |  Scripts)   |     |  Java, etc.)|      |
 *     |    |(Web GUI) |      +-------------+     +-------------+      |
 *     |    +----------+              |                   |            |
 *     |          |                   |                   |            |
 *     |          +-------------------+-------------------+            |
 *     |                              |                                |
 *     |                              V                                |
 *     |                  +---------------------+                      |
 *     |                  |  Same AWS REST APIs |                      |
 *     |                  +---------------------+                      |
 *     |                              |                                |
 *     |                              V                                |
 *     |               +------------------------------+                |
 *     |               | Hypervisor → Physical Layer  |                |
 *     |               +------------------------------+                |
 *     +---------------------------------------------------------------+
 *
 *   Accessing AWS for the First Time (Root Account):
 *   - Go to URL: https://aws.amazon.com/console/
 *   - Since we do not have any IAM user yet, we log in using the
 *     Root User account for the very first time.
 *   - On the login page, click "Sign in using root user email":
 *
 *     +---------------------------------------------------------------+
 *     |                   AWS Console Login                           |
 *     |                                                               |
 *     |   (A) Root User                  (B) IAM User                 |
 *     |   +------------------+           +------------------+         |
 *     |   | - Email Address  |           | - Account ID /   |         |
 *     |   | - Password       |           |   Account Alias  |         |
 *     |   | - MFA (OTP)      |           | - IAM Username   |         |
 *     |   |                  |           | - Password       |         |
 *     |   | Full access to   |           |                  |         |
 *     |   | everything.      |           | Limited access   |         |
 *     |   | Avoid using for  |           | based on IAM     |         |
 *     |   | daily tasks.     |           | policies.        |         |
 *     |   +------------------+           +------------------+         |
 *     |                                                               |
 *     |         [ Sign in using root user email ] (button)            |
 *     |                                                               |
 *     |   +-------------------------+                                 |
 *     |   |  Root User Login        |                                 |
 *     |   |  - Email Address        |  <-- e.g. admin@chaicode.com    |
 *     |   |  - Password             |                                 |
 *     |   |  - MFA / OTP Code       |  <-- if MFA is enabled          |
 *     |   +-------------------------+                                 |
 *     +---------------------------------------------------------------+
 * 
 *   Enable MFA on the Root Account (immediately):
 *   - MFA (Multi-Factor Authentication) adds a second layer of
 *     security on top of your password.
 *   - After entering the password, AWS asks for a time-based OTP
 *     (One-Time Password) from an authenticator app
 *     (e.g., Google Authenticator, Authy).
 *   - Enable MFA on the Root Account before doing anything else.
 *
 *     Login flow with MFA enabled:
 *
 *     [Enter Email / Username]
 *               |
 *               V
 *     [Enter Password]
 *               |
 *               V
 *     [Enter MFA / OTP Code]  <-- 6-digit code from authenticator app
 *               |
 *               V
 *     [Access Granted → AWS Management Console]
 *
 *   Explore the AWS Management Console(after login):
 *   - The AWS Management Console is a web-based graphical UI that
 *     sits on top of the same underlying AWS REST APIs.
 *   - Everything you see and click in the console is ultimately an
 *     API call to AWS under the hood — it is purely a visual layer.
 *
 *     +---------------------------------------------------------------+
 *     |  AWS Management Console                          [Region ▼]   |
 *     +---------------------------------------------------------------+
 *     |  Search Bar:  [ Search for services, features, docs... ]      |
 *     +---------------------------------------------------------------+
 *     |                                                               |
 *     |  Recently Visited         AWS Services (by category)          |
 *     |  +------------------+    +--------------------------------+   |
 *     |  | EC2              |    | Compute:  EC2, Lambda, ECS     |   |
 *     |  | S3               |    | Storage:  S3, EBS, EFS         |   |
 *     |  | RDS              |    | Database: RDS, DynamoDB, Aurora|   |
 *     |  | Lambda           |    | Network:  VPC, Route53, CF     |   |
 *     |  +------------------+    | Monitor:  CloudWatch, X-Ray    |   |
 *     |                          +--------------------------------+   |
 *     |                                                               |
 *     |  Billing Summary          Account Settings                    |
 *     |  +------------------+    +--------------------------------+   |
 *     |  | Current charges  |    | IAM Users, MFA, Billing Alerts |   |
 *     |  | Cost Explorer    |    | Support Plans, Organizations   |   |
 *     |  +------------------+    +--------------------------------+   |
 *     +---------------------------------------------------------------+
 *
 *   Note — AWS CloudShell (top navbar):
 *   - A built-in browser-based CLI terminal inside the console.
 *   - Connects directly to your AWS account — no local setup needed.
 *   - Useful for running quick AWS CLI commands right from the browser.
 *
 *   Note — AWS Regions (top-right dropdown):
 *   - Every resource you create in AWS is tied to a specific Region
 *     (e.g., us-east-1, ap-south-1).
 *   - If you create an EC2 instance in us-east-1 and then switch the
 *     console to ap-south-1, you will NOT see that instance —
 *     because resources are region-scoped.
 *   - Always verify you are in the correct region before creating
 *     or searching for resources.
 *
 *     +-------------------------------+
 *     |  N. Virginia (us-east-1)  ▼  |   <-- currently selected region
 *     +-------------------------------+
 *     |  US East      (us-east-2)     |
 *     |  US West      (us-west-1)     |
 *     |  Asia Pacific (ap-south-1)    |   <-- Mumbai (India)
 *     |  Asia Pacific (ap-southeast-1)|
 *     |  Europe       (eu-west-1)     |
 *     |  ... and more                 |
 *     +-------------------------------+
 */


/**
 * IAM (Identity and Access Management):
 *   - IAM is AWS's built-in service for managing identities (who you are)
 *     and access control (what you are allowed to do).
 *   - It handles User Management, Role Management, Group Management,
 *     and Permission Policies — all under one service.
 * 
 * 
 *   The Root Account:
 *   - When you first create an AWS account, you sign up with an email
 *     address and a password. This creates the Root Account.
 *   - The Root Account has unrestricted access to every AWS service and
 *     resource — including billing, closing the account, and changing
 *     any configuration. It cannot be limited by any policy.
 *   - Because of this, it is strongly advised to NEVER use the root
 *     account for day-to-day tasks. If the root credentials are ever
 *     compromised, the attacker has full control over everything.
 *
 *   Create an IAM User immediately after Root Account setup:
 *
 *     Step 1: Create Root Account
 *             admin@chaicode.com  +  password
 *                       |
 *                       V
 *     Step 2: Log in as Root (only this one time)
 *                       |
 *                       V
 *     Step 3: Go to IAM → Create a new IAM User
 *             hitesh@chaicode.com  +  password  +  Admin permissions
 *                       |
 *                       V
 *     Step 4: Log out of Root Account
 *                       |
 *                       V
 *     Step 5: From now on, always log in as the IAM User
 *             Never touch the Root Account unless absolutely necessary
 *             (e.g., closing the account or recovering access)
 *
 *   IAM User Login — what credentials are needed:
 *   - Unlike root login which only needs an email, an IAM user login
 *     requires three things because multiple IAM users can exist across
 *     millions of AWS accounts — AWS needs to know WHICH account and
 *     WHICH user is logging in.
 *
 *     +---------------------------------------------------------------+
 *     |               IAM User Login Requirements                     |
 *     |                                                               |
 *     |   1. Account ID / Account Alias                               |
 *     |      - A unique 12-digit number assigned to your AWS account  |
 *     |        (e.g., 1234-5678-9012).                                |
 *     |      - Tells AWS WHICH account to log into.                   |
 *     |      - Can be replaced with a human-readable Account Alias    |
 *     |        (e.g., "chaicode") for easier login.                   |
 *     |                                                               |
 *     |   2. IAM Username                                             |
 *     |      - Identifies WHO is logging in within that account.      |
 *     |        (e.g., hitesh@chaicode.com)                            |
 *     |                                                               |
 *     |   3. Password                                                 |
 *     |      - The password set for that IAM user.                    |
 *     +---------------------------------------------------------------+
 *
 *   AWS Management Console (after login):
 *     +---------------------------------------------------------------+
 *     |  AWS Management Console                          [Region ▼]   |
 *     +---------------------------------------------------------------+
 *     |  Search Bar:  [ Search for services, features, docs... ]      |
 *     +---------------------------------------------------------------+
 *     |                                                               |
 *     |  Recently Visited         AWS Services (by category)          |
 *     |  +------------------+    +--------------------------------+   |
 *     |  | EC2              |    | Compute:  EC2, Lambda, ECS     |   |
 *     |  | S3               |    | Storage:  S3, EBS, EFS         |   |
 *     |  | RDS              |    | Database: RDS, DynamoDB, Aurora|   |
 *     |  | Lambda           |    | Network:  VPC, Route53, CF     |   |
 *     |  +------------------+    | Monitor:  CloudWatch, X-Ray    |   |
 *     |                          +--------------------------------+   |
 *     |                                                               |
 *     |  Billing Summary          Account Settings                    |
 *     |  +------------------+    +--------------------------------+   |
 *     |  | Current charges  |    | IAM Users, MFA, Billing Alerts |   |
 *     |  | Cost Explorer    |    | Support Plans, Organizations   |   |
 *     |  +------------------+    +--------------------------------+   |
 *     +---------------------------------------------------------------+
*/


In [ ]:

/**
 * ─────────────────────────────────────────────────
 * Compute at Scale — EC2 (Elastic Compute Cloud):
 * ─────────────────────────────────────────────────
 *   - Now that we are logged in as an IAM user, the next thing we need
 *     is a machine to run our applications on. This is where EC2 comes in.
 *   - EC2 (Elastic Compute Cloud) is AWS's core compute service that
 *     provides resizable virtual machines (called Instances) in the cloud.
 *   - Instead of buying a physical server, you simply spin up an EC2
 *     instance in seconds, use it, and pay only for what you consume.
 * 
 * ─────────────────────────────────────────────────
 *  1. Where EC2 fits in the AWS Stack:
 * ─────────────────────────────────────────────────
 *
 *     +---------------------------------------------------------------+
 *     |                      [Your Application]                       |
 *     +---------------------------------------------------------------+
 *     |                EC2 Instance (Virtual Machine)                 |
 *     |         OS: Ubuntu / Amazon Linux / Windows etc.              |
 *     +---------------------------------------------------------------+
 *     |              Hypervisor (AWS Nitro System)                    |
 *     +---------------------------------------------------------------+
 *     |                      Physical Layer                           |
 *     |         (Servers, Storage, Network, Cooling, Power)           |
 *     +---------------------------------------------------------------+
 * 
 * ─────────────────────────────────────────────────
 * 2. What you are actually renting from AWS:
 * ─────────────────────────────────────────────────
 * 
 *    AWS Data Center (Physical)
 *    ┌──────────────────────────────────────────┐
 *    │  ┌─────────┐  ┌─────────┐  ┌─────────┐   │
 *    │  │ Server 1│  │ Server 2│  │ Server 3│   │
 *    │  │ (yours) │  │ (yours) │  │(someone)│   │
 *    │  │  EC2 #1 │  │  EC2 #2 │  │  else)  │   │
 *    │  └─────────┘  └─────────┘  └─────────┘   │
 *    │       Hypervisor divides physical        │
 *    │      hardware into virtual machines      │
 *    └──────────────────────────────────────────┘
 *                         │
 *                AWS REST APIs / CLI
 *                         │
 *                    [You / Dev]
 * 
 * ─────────────────────────────────────────────────
 * Step 1: Navigate to EC2 in the Console:
 * ─────────────────────────────────────────────────
 *   - Log in to AWS Management Console as your IAM User.
 *   - In the Search Bar, type "EC2" and open it.
 *   - This opens the EC2 Dashboard where you can see all your
 *     compute resources at a glance.
 *
 *   EC2 Dashboard Overview:
 * 
 *     +---------------------------------------------------------------+
 *     |  EC2 Dashboard                              [Region ▼]        |
 *     +---------------------------------------------------------------+
 *     |                                                               |
 *     |  Resources                                                    |
 *     |  +---------------------------+----------------------------+   |
 *     |  | a. Instances (running)    | g. Dedicated Hosts         |   |
 *     |  | b. Capacity Reservations  | h. Instances               |   |
 *     |  | c. Elastic IPs            | i. Load Balancers          |   |
 *     |  | d. Key Pairs              | j. Security Groups         |   |
 *     |  | e. Placement Groups       | k. Volumes (EBS)           |   |
 *     |  | f. Auto Scaling Groups    |                            |   |
 *     |  +---------------------------+----------------------------+   |
 *     +---------------------------------------------------------------+
 * 
 * ─────────────────────────────────────────────────
 * Step 2: Launch a New EC2 Instance:
 * ─────────────────────────────────────────────────
 *   - Click on "Instances" in the left sidebar or dashboard.
 *   - Click the "Launch Instance" button.
 *   - This opens the instance configuration form where we interact
 *     with the Hypervisor APIs to define and provision our machine.
 *
 *   EC2 Instance Configuration — field by field:
 * 
 *   a. Name:
 *      - Give your instance a meaningful name.
 *      - Example: "WebServer", "APIServer", "DBServer"
 *
 *   b. Application and OS Image (AMI — Amazon Machine Image):
 *      - An AMI is a pre-built template that contains the Operating
 *        System and optionally pre-installed software.
 *      - Think of it as a snapshot/image of a machine that AWS uses
 *        to boot your instance.
 *      - Common choices:
 * 
 *        +-----------------------------------------------------------+
 *        |  AMI Options                                              |
 *        +---------------------+-------------------------------------+
 *        |  Amazon Linux 2     |  AWS's own lightweight Linux.       |
 *        |                     |  Best for AWS-native workloads.     |
 *        +---------------------+-------------------------------------+
 *        |  Ubuntu             |  Most popular for web servers and   |
 *        |                     |  general-purpose development.       |
 *        +---------------------+-------------------------------------+
 *        |  Windows Server     |  For .NET apps or Windows-based     |
 *        |                     |  workloads.                         |
 *        +---------------------+-------------------------------------+
 *        |  macOS              |  For iOS/macOS app build pipelines. |
 *        +---------------------+-------------------------------------+
 *        |  Custom AMI         |  Your own saved machine snapshot.   |
 *        |                     |  Useful for replicating servers.    |
 *        +---------------------+-------------------------------------+
 * 
 *        Existing Machine (configured)
 *        ┌──────────────────────────┐
 *        │  OS: Ubuntu 22.04        │
 *        │  NGINX installed         │──── Snapshot ────► AMI
 *        │  Node.js installed       │                  (stored in S3)
 *        │  App configs set         │
 *        └──────────────────────────┘
 *
 *        Later — Launch 5 identical machines from same AMI:
 *
 *                    [AMI]
 *                   /  |  \
 *                  /   |   \
 *               EC2#1 EC2#2 EC2#3   ← all identical clones
 * 
 *   c. Instance Type:
 *      - Defines the hardware specs of your virtual machine —
 *        how much CPU and RAM it gets.
 *      - AWS instance types follow this naming pattern:
 *
 *        [Family][Generation].[Size]
 *        e.g., t3.micro, m5.large, c6i.xlarge
 *
 *        +-------------+--------+-------+----------------------------+
 *        |  Type        | vCPUs  |  RAM  |  Use Case                 |
 *        +-------------+--------+-------+----------------------------+
 *        |  t3.micro   |   2    |  1 GB |  Free tier, testing, dev  |
 *        |  t3.small   |   2    |  2 GB |  Small web apps            |
 *        |  t3.medium  |   2    |  4 GB |  Medium workloads          |
 *        |  m5.large   |   2    |  8 GB |  General purpose apps      |
 *        |  c5.large   |   2    |  4 GB |  CPU-intensive workloads   |
 *        |  r5.large   |   2    | 16 GB |  Memory-intensive workloads|
 *        +-------------+--------+-------+----------------------------+
 *
 *      - For learning and free tier: always use t2.micro or t3.micro.
 *        t  3  .  micro
 *        │  │     │
 *        │  │     └─ Size: nano / micro / small / medium / large /
 *        │  │                xlarge / 2xlarge ...
 *        │  └─ Generation: higher = newer & better
 *        └─ Family:
 *             t = Burstable (general, cheap)
 *             m = General Purpose
 *             c = Compute Optimized (CPU heavy)
 *             r = Memory Optimized (RAM heavy)
 *             g = GPU Instances (ML, graphics)
 *             i = Storage Optimized (fast disk I/O) 
 * 
 *   d. Key Pair (Login):
 *      - A Key Pair is used to securely SSH into your EC2 instance.
 *      - AWS stores the Public Key on the instance.
 *      - You download and keep the Private Key (.pem file) locally.
 *      - To connect:  ssh -i "your-key.pem" ubuntu@<public-ip>
 *      - IMPORTANT: If you lose the .pem file, you lose access to
 *        that instance. There is no way to recover it.
 *
 *        +-------------------------------------------------------+
 *        |  Key Pair — How it works                              |
 *        |                                                       |
 *        |  [Your Machine]          [EC2 Instance]               |
 *        |  Private Key (.pem)  →   Public Key (stored by AWS)   |
 *        |                                                       |
 *        |  SSH connection is established only when both         |
 *        |  keys match. No password needed.                      |
 *        +-------------------------------------------------------+
 * 
 *   e. Network Settings — Security Group:
 *      - A Security Group acts as a virtual firewall for your
 *        EC2 instance, controlling inbound and outbound traffic.
 *      - You define rules specifying which ports are open and
 *        from which IP addresses traffic is allowed.
 *
 *        Common inbound rules:
 *
 *        +----------+-------+-------------------+------------------+
 *        |  Type    | Port  |  Source           |  Purpose         |
 *        +----------+-------+-------------------+------------------+
 *        |  SSH     |  22   |  Your IP / 0.0.0.0|  Remote terminal |
 *        |  HTTP    |  80   |  0.0.0.0/0        |  Web traffic     |
 *        |  HTTPS   |  443  |  0.0.0.0/0        |  Secure web      |
 *        |  Custom  |  3000 |  0.0.0.0/0        |  Node.js app     |
 *        |  Custom  |  5432 |  Your IP only     |  PostgreSQL DB   |
 *        +----------+-------+-------------------+------------------+
 *
 *                         Internet
 *                            │
 *               ┌────────────▼─────────────┐
 *               │     Security Group       │
 *               │  ┌─────────────────────┐ │
 *               │  │ Inbound Rules:      │ │
 *               │  │ ✓ Port 22 (my IP)   │ │  ← only you can SSH
 *               │  │ ✓ Port 80 (anyone)  │ │  ← everyone can browse
 *               │  │ ✗ Port 5432 (block) │ │  ← DB hidden from web
 *               │  └─────────────────────┘ │
 *               └────────────┬─────────────┘
 *                            │ allowed traffic only
 *                            ▼
 *                     ┌─────────────┐
 *                     │ EC2 Instance│
 *                     └─────────────┘
 * 
 *      - Best practice: never open port 22 (SSH) to 0.0.0.0/0
 *        (the entire internet). Restrict it to your IP only.
 * 
 *   f. Storage (EBS — Elastic Block Store):
 *      - Every EC2 instance gets a root EBS volume (the hard disk).
 *      - Default: 8 GB for Amazon Linux / Ubuntu.
 *      - EBS volumes persist independently of the instance — meaning
 *        if you stop the instance, the data on EBS is NOT lost.
 *      - You can attach multiple EBS volumes to one instance.
 *
 *        +-------------------------------------------------------+
 *        |  Storage Options                                      |
 *        +------------------+------------------------------------+
 *        |  gp3 (General    |  Default. Good for most workloads. |
 *        |  Purpose SSD)    |  Cost-effective and fast.          |
 *        +------------------+------------------------------------+
 *        |  io2 (Provisioned|  High-performance. For databases   |
 *        |  IOPS SSD)       |  needing low latency.              |
 *        +------------------+------------------------------------+
 *        |  st1 (HDD)       |  Large sequential workloads like   |
 *        |                  |  log processing or data warehouses.|
 *        +------------------+------------------------------------+
 * 
 *      VISUAL — EBS vs Instance Storage:
 *
 *        ┌──────────────────────────────────────┐
 *        │           EC2 Instance               │
 *        │   ┌─────────────────────────────┐    │
 *        │   │  Instance Store (temp disk) │    │
 *        │   │  Fast but LOST on stop/term │    │
 *        │   └─────────────────────────────┘    │
 *        └──────────────────┬───────────────────┘
 *                           │ attached via network
 *                           ▼
 *        ┌──────────────────────────────────────┐
 *        │       EBS Volume (persistent)        │
 *        │   Survives stop / reboot / detach    │
 *        │   Can be detached and re-attached    │
 *        │    to a different EC2 instance       │
 *        └──────────────────────────────────────┘
 * 
 *   g. Summary before launch:
 *     +---------------------------------------------------------------+
 *     |  Launch Instance — Summary                                    |
 *     |                                                               |
 *     |  Name          :  WebServer                                   |
 *     |  AMI           :  Ubuntu 22.04 LTS                            |
 *     |  Instance Type :  t3.micro (Free tier eligible)               |
 *     |  Key Pair      :  my-key.pem  (downloaded to local machine)   |
 *     |  Security Group:  Allow SSH (port 22), HTTP (80), HTTPS (443) |
 *     |  Storage       :  8 GB gp3 EBS Volume                         |
 *     |                                                               |
 *     |                    [ Launch Instance ]                        |
 *     +---------------------------------------------------------------+
 * 
 * ─────────────────────────────────────────────────
 * 3. Connect to Your EC2 Instance:
 * ─────────────────────────────────────────────────
 *   - Once the instance is running (state: "running"), you can
 *     connect to it in two ways:
 *   
 *     a. EC2 Instance Connect (browser-based):
 *        - Select your instance → click "Connect" → "EC2 Instance Connect"
 *        - Opens a terminal directly in the browser — no .pem file needed.
 * 
 *     b. SSH from your local terminal (local-to-cloud):
 *        - chmod 400 my-key.pem
 *        - ssh -i "my-key.pem" ubuntu@<EC2-Public-IP>
 * 
 * 
 * ─────────────────────────────────────────────────
 * 4. Set Up a Web Server on the EC2 Instance:
 * ─────────────────────────────────────────────────
 *   - Once connected via SSH or EC2 Instance Connect, the machine
 *     is completely empty — we need to install software on it.
 *   - We will install NGINX as a web server to verify the instance
 *     is publicly accessible.
 * 
 * 
 *   Run the following commands inside the EC2 terminal:
 *    a. Update and upgrade the package list:
 *       $ sudo apt update
 *       $ sudo apt upgrade -y
 *
 *    b. Install NGINX:
 *       $ sudo apt install nginx -y
 *
 *    c. Verify NGINX is running:
 *       $ sudo systemctl status nginx
 *       -- Output should show: Active: active (running)
 *
 *    d. Test locally on the instance:
 *       $ curl http://localhost:80
 *       -- NGINX default page HTML should be returned.
 *
 *    e. Try accessing from your browser using the EC2 Public IP:
 *       http://<EC2-Public-IP>
 *       -- This will NOT load yet — the browser request is being
 *          blocked by the Security Group firewall.
 * 
 *   What NGINX does on the EC2 instance:
 *
 *                 EC2 Instance
 *     ┌────────────────────────────────────┐
 *     │                                    │
 *     │           NGINX process            │
 *     │   ┌─────────────────────────────┐  │
 *     │   │    Listening on port 80     │  │
 *     │   │    Serves HTML files /      │  │
 *     │   │    Proxies to app server    │  │
 *     │   └──────────────┬──────────────┘  │
 *     │                  │                 │
 *     └──────────────────┼─────────────────┘
 *                        │
 *         ◄──── request  │  response ────►
 *                        │
 *                 [Port 80 exposed]
 *            (if Security Group allows it)
 * 
 * ─────────────────────────────────────────────────
 * 5. Fix the Security Group to Allow HTTP Traffic:
 * ─────────────────────────────────────────────────
 * 
 *   - NGINX is running on port 80 inside the instance, but the
 *     Security Group attached to this instance does not have an
 *     inbound rule allowing traffic on port 80 from the internet.
 *   - We need to add an inbound rule to allow HTTP traffic.
 * 
 *   How to update the Security Group:
 * 
 *     EC2 Dashboard
 *       → Select your instance
 *       → Click "Security" tab
 *       → Click on the Security Group name
 *       → Click "Edit Inbound Rules"
 *       → Click "Add Rule"
 *           Type        : Custom TCP  (or HTTP)
 *           Port Range  : 80
 *           Source      : Anywhere - IPv4  (0.0.0.0/0)
 *       → Click "Save Rules"
 * 
 *   Security Group — Before and After:
 *     +---------------------------------------------------------------+
 *     |  Inbound Rules — BEFORE          Inbound Rules — AFTER        |
 *     +------------------------+--------------------------------------+
 *     |  SSH  | 22 | Your IP   |  SSH  | 22 | Your IP   | ✓ Open      |
 *     |                        |  HTTP | 80 | 0.0.0.0/0 | ✓ Added     |
 *     +------------------------+--------------------------------------+
 *
 *   - Now open your browser and visit:
 *     http://<EC2-Public-IP>
 *   - You should see the NGINX default welcome page — your EC2
 *     instance is now publicly accessible on the internet.
 * 
 *   How traffic flows end to end:
 * 
 *     [User Browser]
 *           |
 *           | HTTP request on port 80
 *           V
 *     [Security Group — Inbound Rule: port 80 allowed]
 *           |
 *           V
 *     [EC2 Instance — NGINX listening on port 80]
 *           |
 *           V
 *     [NGINX serves response back to the browser]
 * 
 *
 *   Note: Once you start using Docker, you no longer need to install
 *   NGINX directly on the EC2 instance. Instead, NGINX (or any other
 *   server) will run inside a Docker container on the same machine.
 * 
 * 
 * ───────────────────────────────────────────────────────
 * 6. The IP Address Problem — Why It Changes Every Time:
 * ───────────────────────────────────────────────────────
 * 
 *   - Every EC2 instance gets a Public IP address when it starts.
 *   - But this Public IP is a DYNAMIC IP — meaning it changes
 *     every time you stop and restart the instance.
 *   - AWS draws from a shared pool of IPs and assigns a free one
 *     to your instance at launch. When you stop the instance,
 *     that IP is released back into the pool for others to use.
 *     When you start it again, you get a different IP.
 *
 *   The problem this creates:
 *
 *     Day 1: Instance starts → Public IP = 54.23.11.5
 *               |
 *               | You note down this IP and share it with your team
 *               |
 *     [Stop instance at night to save cost]
 *               |
 *     Day 2: Instance restarts → Public IP = 18.97.44.2  ← CHANGED!
 *               |
 *               | Your SSH command fails.
 *               | Your domain or config pointing to old IP breaks.
 *               | Your team cannot reach the server.
 *
 *     +---------------------------------------------------------------+
 *     |  Dynamic IP — What happens on Stop/Start                      |
 *     |                                                               |
 *     |  [EC2 Running]  IP: 54.23.11.5                                |
 *     |        |                                                      |
 *     |        V                                                      |
 *     |  [EC2 Stopped]  IP: released back to AWS pool                 |
 *     |        |                                                      |
 *     |        V                                                      |
 *     |  [EC2 Started]  IP: 18.97.44.2  <-- NEW IP assigned           |
 *     |                                                               |
 *     |  Result: SSH breaks, DNS breaks, configs break.               |
 *     +---------------------------------------------------------------+
 *
 *   VISUAL — The dynamic IP problem over time:
 *
 *    Monday                  Tuesday                 Wednesday
 *    ┌───────────────┐       ┌───────────────┐       ┌───────────────┐
 *    │ EC2 Running   │  stop │ EC2 Stopped   │ start │ EC2 Running   │
 *    │ IP:54.23.11.5 │──────►│ IP: released  │──────►│ IP:18.97.44.2 │
 *    └───────────────┘       └───────────────┘       └───────────────┘
 *          │                                                │
 *          │ Team bookmarks                                 │
 *          │ 54.23.11.5 ✓                                   │
 *          │                                   54.23.11.5 ✗ broken
 *          │                                   18.97.44.2 ✓ new IP
 *          │                                   (but nobody knows it)
 *          ▼
 *    App accessible                             App inaccessible ✗
 * 
 * ───────────────────────────────────────────────────────
 * 7. Solution A: Elastic IP (Static Public IP)
 * ───────────────────────────────────────────────────────
 * 
 *   - An Elastic IP is a STATIC, fixed public IP address that you
 *     own and reserve in your AWS account.
 *   - It does not change on stop/start — it stays yours until
 *     you explicitly release it.
 *   - You attach it to your EC2 instance and it stays fixed
 *     regardless of how many times you stop and restart.
 *
 *     +---------------------------------------------------------------+
 *     |  Elastic IP — How it works                                    |
 *     |                                                               |
 *     |  AWS Account                                                  |
 *     |  +------------------+                                         |
 *     |  | Elastic IP       |  54.23.11.5  (reserved, yours forever)  |
 *     |  +------------------+                                         |
 *     |          |                                                    |
 *     |          | attached to                                        |
 *     |          V                                                    |
 *     |  +------------------+                                         |
 *     |  | EC2 Instance     |  Stop → Start → IP stays 54.23.11.5     |
 *     |  +------------------+                                         |
 *     |                                                               |
 *     |  IMPORTANT: Elastic IPs are FREE only when attached to a      |
 *     |  running instance. If you reserve one but do NOT attach it    |
 *     |  (or the instance is stopped), AWS charges you for it.        |
 *     +---------------------------------------------------------------+
 * 
 *   VISUAL — Dynamic IP vs Elastic IP comparison:
 *
 *    WITHOUT Elastic IP          WITH Elastic IP
 *    ┌─────────────────┐         ┌─────────────────┐
 *    │ EC2 (running)   │         │ EC2 (running)   │
 *    │ IP: 54.23.11.5  │         │ IP: 54.23.11.5  │◄─┐
 *    └────────┬────────┘         └────────┬────────┘  │
 *             │ stop                      │ stop      │ Elastic IP
 *             ▼                           ▼           │ 54.23.11.5
 *    ┌─────────────────┐         ┌─────────────────┐  │ stays reserved
 *    │ EC2 (stopped)   │         │ EC2 (stopped)   │  │
 *    │ IP: released ✗  │         │ IP: still yours │──┘
 *    └────────┬────────┘         └────────┬────────┘
 *             │ start                     │ start
 *             ▼                           ▼
 *    ┌─────────────────┐         ┌─────────────────┐
 *    │ EC2 (running)   │         │ EC2 (running)   │
 *    │ IP: 18.97.44.2  │         │ IP: 54.23.11.5  │ ← same IP ✓
 *    │ CHANGED ✗       │         └─────────────────┘
 *    └─────────────────┘
 *
 * 
 *   PART A — Allocate (Reserve) an Elastic IP:
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Elastic IPs"
 *       → Click "Allocate Elastic IP address"
 *       → Network Border Group: select your Region
 *           (e.g., ap-south-1 for Mumbai)
 *       → Click "Allocate"
 *       → AWS reserves a static IP for your account
 *           e.g., 54.23.11.5  ← now yours until you release it
 *
 *   PART B — Attach (Associate) the Elastic IP to an EC2 Instance:
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Elastic IPs"
 *       → Select the allocated Elastic IP
 *       → Click "Actions" → "Associate Elastic IP address"
 *       → Resource type : Instance
 *       → Instance      : select your EC2 instance (e.g., WebServer)
 *       → Private IP    : select the private IP of the instance
 *       → Click "Associate"
 *
 *     After association:
 *
 *     +---------------------------------------------------------------+
 *     |  EC2 Instance — Elastic IP attached                           |
 *     |                                                               |
 *     |  Instance ID   :  i-0abc123def456                             |
 *     |  Private IP    :  172.31.5.10   (internal, never changes)     |
 *     |  Public IP     :  54.23.11.5    (Elastic IP, fixed forever)   |
 *     |                                                               |
 *     |  Stop  → Start → Public IP still 54.23.11.5  ✓                |
 *     +---------------------------------------------------------------+
 * 
 * 
 *   PART C — Detach (Disassociate) an Elastic IP:
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Elastic IPs"
 *       → Select the Elastic IP
 *       → Click "Actions" → "Disassociate Elastic IP address"
 *       → Confirm → Click "Disassociate"
 *
 *     After disassociation:
 *     - The Elastic IP is no longer attached to any instance.
 *     - The EC2 instance loses its fixed public IP.
 *     - AWS will now CHARGE you for the unattached Elastic IP.
 *     - You must either re-attach it or release it immediately.
 *
 *   PART D — Release (Delete) an Elastic IP:
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Elastic IPs"
 *       → Select the Elastic IP  (must be disassociated first)
 *       → Click "Actions" → "Release Elastic IP address"
 *       → Confirm → Click "Release"
 *
 *     After release:
 *     - The IP is returned to AWS's shared pool.
 *     - You no longer own it — someone else may get this IP.
 *     - Billing for this IP stops immediately.
 * 
 * 
 *   Full Elastic IP Lifecycle:
 *
 *     [Allocate]     → IP reserved, yours, not yet attached
 *         |             (AWS charges if left unattached)
 *         V
 *     [Associate]    → IP attached to running EC2 instance
 *         |             (Free as long as instance is running)
 *         V
 *     [Disassociate] → IP detached from instance
 *         |             (AWS charges again — unattached)
 *         V
 *     [Release]      → IP returned to AWS pool
 *                       (Billing stops, IP is gone forever)
 *
 *   Billing rules to remember:
 *   - 1 Elastic IP attached to 1 running instance  → FREE
 *   - Elastic IP allocated but NOT attached         → CHARGED
 *   - Elastic IP attached to a STOPPED instance     → CHARGED
 *   - Multiple unattached Elastic IPs               → CHARGED per IP
 *
 *
 *   VISUAL — Elastic IP billing summary:
 *
 *    ┌─────────────────────────────────────────────────────────┐
 *    │  Scenario                              Billing          │
 *    ├─────────────────────────────────────────────────────────┤
 *    │  Elastic IP + Running Instance    →   FREE  ✓           │
 *    │  Elastic IP + Stopped Instance    →   CHARGED  ✗        │
 *    │  Elastic IP with no Instance      →   CHARGED  ✗        │
 *    │  No Elastic IP at all             →   FREE (dynamic IP) │
 *    └─────────────────────────────────────────────────────────┘
 *
 *    Rule of thumb: if you're not using it → release it.
 * 
 * 
 * ───────────────────────────────────────────────────────
 * 8. Solution B — Elastic IP + DNS (Domain Name System)
 * ───────────────────────────────────────────────────────
 * 
 *   - Instead of sharing a raw IP like 54.23.11.5 with users,
 *     you point a domain name (e.g., api.chaicode.com) to your
 *     EC2 Elastic IP using DNS records.
 *   - This way, users always visit api.chaicode.com — they never
 *     need to know or care about the underlying IP address.
 *
 *   How DNS works (concept):
 *
 *     Think of DNS like a phone book:
 *     - You don't memorize phone numbers, you look up a name.
 *     - DNS does the same — you type a domain name, DNS looks
 *       up the IP address behind it and connects you.
 *
 *     [User types chaicode.com in browser]
 *               |
 *               V
 *     [Browser checks its local DNS cache]
 *               |
 *               | (not found / expired)
 *               V
 *     [ISP's DNS Resolver asks the Root DNS Server]
 *               |
 *               V
 *     [Root DNS Server says: .com is handled by Verisign]
 *               |
 *               V
 *     [Verisign TLD Server says: chaicode.com is at GoDaddy]
 *               |
 *               V
 *     [GoDaddy DNS Server returns: chaicode.com → 54.23.11.5]
 *               |
 *               V
 *     [Browser connects to 54.23.11.5 → EC2 Instance]
 *               |
 *               V
 *     [NGINX serves the response back to the user]
 * 
 * 
 *   VISUAL — Full DNS resolution chain:
 *
 *    You type: chaicode.com
 *         │
 *         ▼
 *    ┌──────────────────┐
 *    │  Browser Cache   │──── found? ──────────────────────────► EC2
 *    └────────┬─────────┘
 *             │ not found
 *             ▼
 *    ┌──────────────────┐
 *    │  ISP Resolver    │  (your internet provider's DNS server)
 *    └────────┬─────────┘
 *             │
 *             ▼
 *    ┌──────────────────┐
 *    │  Root DNS (.)    │  "who handles .com?"
 *    └────────┬─────────┘
 *             │
 *             ▼
 *    ┌──────────────────┐
 *    │  TLD Server      │  "chaicode.com is at GoDaddy"
 *    │  (.com — Verisign│
 *    └────────┬─────────┘
 *             │
 *             ▼
 *    ┌──────────────────┐
 *    │  GoDaddy DNS     │  A Record: chaicode.com → 54.23.11.5
 *    └────────┬─────────┘
 *             │
 *             ▼
 *         54.23.11.5
 *             │
 *             ▼
 *    ┌──────────────────┐
 *    │  EC2 Instance    │  NGINX serves response ✓
 *    └──────────────────┘
 *
 * 
 *   DNS Record Types used with EC2:
 * 
 *     +--------+--------------------------------------------------+
 *     |  Type  |  Description                                     |
 *     +--------+--------------------------------------------------+
 *     |  A     |  Maps a domain name directly to an IPv4 address. |
 *     |        |  e.g., chaicode.com → 54.23.11.5                 |
 *     +--------+--------------------------------------------------+
 *     |  CNAME |  Maps a domain name to another domain name.      |
 *     |        |  e.g., www.chaicode.com → chaicode.com           |
 *     +--------+--------------------------------------------------+
 *     |  AAAA  |  Maps a domain name to an IPv6 address.          |
 *     +--------+--------------------------------------------------+
 *
 * 
 *   How to Point a GoDaddy Domain to an EC2 Instance
 *
 *   Prerequisites:
 *   - You have bought a domain on GoDaddy (e.g., chaicode.com).
 *   - Your EC2 instance has an Elastic IP attached (e.g., 54.23.11.5).
 *     Always use an Elastic IP before pointing a domain — otherwise
 *     the IP will change on stop/start and your domain will break.
 *
 *   PART A — Log in to GoDaddy and open DNS Settings:
 *     1. Go to https://www.godaddy.com and log in.
 *     2. Click on your profile → "My Products".
 *     3. Find your domain (e.g., chaicode.com).
 *     4. Click "DNS" or "Manage DNS" next to the domain.
 *     5. This opens the DNS Management page where all your
 *        DNS records are listed.
 *
 *     GoDaddy DNS Management Page:
 *     +---------------------------------------------------------------+
 *     |  DNS Management — chaicode.com                                |
 *     +--------+---------+------------------+------------------------+|
 *     |  Type  |  Name   |  Value           |  TTL                   ||
 *     +--------+---------+------------------+------------------------+|
 *     |  A     |  @      |  Parked IP       |  1 Hour                ||
 *     |  CNAME |  www    |  @               |  1 Hour                ||
 *     +--------+---------+------------------+------------------------+|
 *     |                              [ Add New Record ]               ||
 *     +---------------------------------------------------------------+
 *
 *   PART B — Add an A Record to point to your EC2 Elastic IP:
 *
 *     Click "Edit" on the existing A record or "Add New Record":
 *     +---------------------------------------------------------------+
 *     |  Add / Edit DNS Record                                        |
 *     |                                                               |
 *     |  Type  :  A                                                   |
 *     |  Name  :  @                                                   |
 *     |           (@ means the root domain itself — chaicode.com)     |
 *     |  Value :  54.23.11.5    <-- your EC2 Elastic IP               |
 *     |  TTL   :  1 Hour        <-- how long DNS cache is kept        |
 *     |                                                               |
 *     |                          [ Save ]                             |
 *     +---------------------------------------------------------------+
 *
 *   PART C — Add a subdomain A Record (optional but common):
 *
 *     If you want api.chaicode.com to point to the same EC2 instance:
 *     +---------------------------------------------------------------+
 *     |  Add DNS Record                                               |
 *     |                                                               |
 *     |  Type  :  A                                                   |
 *     |  Name  :  api   (this creates api.chaicode.com)               |
 *     |  Value :  54.23.11.5    <-- same EC2 Elastic IP               |
 *     |  TTL   :  1 Hour                                              |
 *     |                                                               |
 *     |                          [ Save ]                             |
 *     +---------------------------------------------------------------+
 *
 *     After saving, your DNS records will look like this:
 *     +--------+---------+-----------------+--------+---------------+
 *     |  Type  |  Name   |  Value          |  TTL   |  Resolves to  |
 *     +--------+---------+-----------------+--------+---------------+
 *     |  A     |  @      |  54.23.11.5     |  1 Hr  |  chaicode.com |
 *     |  A     |  api    |  54.23.11.5     |  1 Hr  |  api.chai...  |
 *     |  CNAME |  www    |  @              |  1 Hr  |  www.chai...  |
 *     +--------+---------+-----------------+--------+---------------+
 *
 *   PART D — Wait for DNS Propagation:
 *   - After saving DNS records, changes do NOT take effect
 *     instantly — this is called DNS Propagation.
 *   - DNS servers around the world need time to update their
 *     cache with your new records.
 *   - Propagation typically takes a few minutes to 48 hours
 *     depending on the TTL value and the DNS provider.
 *
 *     +---------------------------------------------------------------+
 *     |  DNS Propagation — What happens                               |
 *     |                                                               |
 *     |  You save the A Record on GoDaddy                             |
 *     |         |                                                     |
 *     |         V                                                     |
 *     |  GoDaddy DNS updated immediately                              |
 *     |         |                                                     |
 *     |         V                                                     |
 *     |  Other DNS servers around the world start updating            |
 *     |  (this can take a few minutes to 48 hours)                    |
 *     |         |                                                     |
 *     |         V                                                     |
 *     |  Old cached IP expires (based on TTL value)                   |
 *     |         |                                                     |
 *     |         V                                                     |
 *     |  All users now resolve chaicode.com → 54.23.11.5  ✓          |
 *     +---------------------------------------------------------------+
 *
 *   - To check if propagation is complete, you can use:
 *     https://dnschecker.org  — shows propagation status worldwide.
 *     $ nslookup chaicode.com — checks DNS resolution from terminal.
 *     $ dig chaicode.com      — detailed DNS lookup from terminal.
 * 
 *   VISUAL — What breaks without Elastic IP + DNS together:
 *
 *    ┌──────────────────────────────────────────────────────────┐
 *    │  GoDaddy: chaicode.com → 54.23.11.5 (A record saved)    │
 *    └──────────────────────┬───────────────────────────────────┘
 *                           │
 *    EC2 restarts           │
 *    New IP: 18.97.44.2     │
 *                           │
 *    User visits chaicode.com
 *                           │
 *                           ▼
 *                   DNS returns 54.23.11.5
 *                           │
 *                           ▼
 *                 ✗ nobody home at 54.23.11.5
 *                   (connection refused / timeout)
 *
 *    Fix:  GoDaddy A Record → update to 18.97.44.2 manually
 *          (painful every restart — use Elastic IP instead)
 *
 * ───────────────────────────────────────────────────────
 * 9. SSL Certificate (Securing with HTTPS):
 * ───────────────────────────────────────────────────────
 * 
 *   - Your site is now live at chaicode.com — but it runs on HTTP.
 *   - HTTP sends all data as plain text. Anyone between the user
 *     and your server can read passwords, form data, everything.
 *   - HTTPS encrypts all data in transit using SSL/TLS.
 *   - Browsers show a 🔒 padlock for HTTPS and a "Not Secure"
 *     warning for HTTP — users will not trust your site without it.
 *
 *
 *   HTTP vs HTTPS — what the difference looks like:
 *
 *    HTTP (anyone can intercept and read):
 *    [Browser] ──── POST /login, password=secret123 ────► [Server]
 *                          ↑ hacker reads this ✗
 *
 *    HTTPS (encrypted — unreadable):
 *    [Browser] ──── xK9#mP2$qR7&nL4... (encrypted) ────► [Server]
 *                          ↑ hacker sees gibberish ✓
 * 
 *   What is AWS ACM (Certificate Manager)?
 *   - ACM is a free AWS service that issues and manages SSL
 *     certificates for AWS resources like ALB and CloudFront.
 *   - Certificates are completely FREE from ACM.
 *   - ACM auto-renews certificates before they expire — you
 *     never have to manually renew.
 *   - ACM certificates cannot be installed directly on EC2.
 *     They only attach to AWS services (ALB, CloudFront, etc.)
 *   - This is why we terminate SSL at the ALB — the ALB handles
 *     encryption/decryption and forwards plain HTTP to EC2.
 *
 *
 *   PART A — Request ACM Certificate:
 *
 *     IMPORTANT region rules:
 *     - For ALB        → request cert in SAME region as ALB
 *                         e.g., ap-south-1 for Mumbai
 *     - For CloudFront → MUST request in us-east-1 (N. Virginia)
 *                         CloudFront only reads certs from us-east-1
 *
 *     AWS Console
 *       → Search "Certificate Manager" → open ACM
 *       → Confirm you are in the correct region (top right)
 *       → Click "Request a certificate"
 *       → Select: "Request a public certificate" → Next
 *       → Domain names:
 *           chaicode.com        ← root domain
 *           *.chaicode.com      ← wildcard (covers ALL subdomains)
 *       → Validation method: DNS validation (recommended)
 *       → Click "Request"
 *
 *     Adding both covers:
 *     ✓ chaicode.com
 *     ✓ www.chaicode.com
 *     ✓ api.chaicode.com
 *     ✓ admin.chaicode.com  (any subdomain)
 * 
 * 
 *   PART B — Validate domain ownership via GoDaddy:
 *
 *     ACM will show you a CNAME record to prove you own the domain:
 *
 *     +---------------------------------------------------------------+
 *     |  ACM DNS Validation Record                                    |
 *     |  Name  : _abc123def456.chaicode.com                          |
 *     |  Type  : CNAME                                                |
 *     |  Value : _xyz789.acm-validations.aws.                        |
 *     +---------------------------------------------------------------+
 *
 *     Add this to GoDaddy:
 *       → GoDaddy → My Products → chaicode.com → Manage DNS
 *       → Add New Record
 *           Type  : CNAME
 *           Name  : _abc123def456   (part before .chaicode.com)
 *           Value : _xyz789.acm-validations.aws.
 *           TTL   : 1 Hour
 *       → Save
 *
 *     Wait for ACM status to change:
 *     Pending validation → Issued ✓  (takes 5–30 minutes)
 *
 *     Keep this CNAME in GoDaddy forever → ACM auto-renews ✓
 * 
 * 
 *   PART C — Attach Certificate to ALB (HTTPS Listener):
 *
 *     EC2 Console → Load Balancers → select your ALB
 *       → "Listeners and rules" tab → "Add listener"
 *           Protocol         : HTTPS
 *           Port             : 443
 *           Default action   : Forward to your Target Group
 *           Certificate      : select chaicode.com from ACM
 *       → Click "Add"
 *
 *
 *   PART D — Redirect HTTP → HTTPS:
 *
 *     EC2 Console → ALB → Listeners
 *       → Select HTTP:80 → "Edit listener"
 *           Default action : Redirect to HTTPS:443
 *           Status code    : 301 (Permanent)
 *       → Save
 *
 *     ALB listeners after setup:
 *     +---------------------------------------------------------------+
 *     |  HTTP  : 80  → 301 Redirect to HTTPS:443                      |
 *     |  HTTPS : 443 → Forward to Target Group (ACM cert attached)    |
 *     +---------------------------------------------------------------+
 * 
 * 
 *   VISUAL — Full HTTPS flow after setup:
 *
 *    User types http://chaicode.com
 *         │
 *         ▼
 *    [ALB port 80] → 301 redirect → https://chaicode.com
 *         │
 *         ▼
 *    [ALB port 443] → TLS handshake using ACM certificate
 *    SSL terminates here — EC2 only receives plain HTTP internally
 *         │
 *         ▼
 *    [EC2 Instance — NGINX]
 *    No SSL config needed on EC2 ✓
 *         │
 *         ▼
 *    User sees 🔒 https://chaicode.com ✓
 *
 *
 *   PART E — Verify HTTPS is working:
 *
 *     → Open https://chaicode.com → padlock shows ✓
 *     → Open http://chaicode.com  → auto-redirects to HTTPS ✓
 *     → Run SSL test: https://www.ssllabs.com/ssltest/
 *       Should show Grade A or A+ ✓
 *     → Terminal: $ curl -I https://chaicode.com
 *       Should show: HTTP/2 200 ✓
 * 
 *   Common mistakes:
 *
 *     +-------------------------------+--------------------------------+
 *     |  Mistake                      |  Fix                           |
 *     +-------------------------------+--------------------------------+
 *     |  Cert stuck "Pending"         |  Check CNAME in GoDaddy is     |
 *     |                               |  correct. Takes up to 30 min.  |
 *     +-------------------------------+--------------------------------+
 *     |  Cert not in ALB dropdown     |  Cert must be in SAME region   |
 *     |                               |  as ALB. Request a new one.    |
 *     +-------------------------------+--------------------------------+
 *     |  Cert not in CloudFront       |  CloudFront needs cert in      |
 *     |                               |  us-east-1 specifically.       |
 *     +-------------------------------+--------------------------------+
 *     |  Padlock not showing          |  Mixed content — some assets   |
 *     |                               |  still load over HTTP. Fix     |
 *     |                               |  all URLs to use HTTPS.        |
 *     +-------------------------------+--------------------------------+
 *
 * ───────────────────────────────────────────────────────
 * 10. EC2 Instance Lifecycle:
 * ───────────────────────────────────────────────────────
 * 
 *    +---------------------------------------------------------------+
 *    |  EC2 Instance States                                          |
 *    |                                                               |
 *    |   [Launch Instance]                                           |
 *    |         |                                                     |
 *    |         V                                                     |
 *    |     [Pending]  <-- instance is being provisioned by AWS       |
 *    |         |                                                     |
 *    |         V                                                     |
 *    |     [Running]  <-- instance is live, billing starts           |
 *    |      IP assigned (dynamic unless Elastic IP is attached)      |
 *    |         |                                                     |
 *    |    +----+-----------+----------+                              |
 *    |    |                |          |                              |
 *    |    V                V          V                              |
 *    | [Stopped]       [Rebooted]  [Terminated]                      |
 *    |    |                |          |                              |
 *    | Not billed for   Restarts   Instance permanently              |
 *    | compute. EBS     the same   deleted. EBS root volume          |
 *    | storage still    instance.  also deleted by default.          |
 *    | costs.           IP stays.  Elastic IP must be released       |
 *    | Dynamic IP                  manually to avoid charges.        |
 *    | is RELEASED.                (cannot recover)                  |
 *    |    |                               |                          |
 *    |    V                               X                          |
 *    | [Running again]                                               |
 *    | NEW dynamic IP assigned                                       |
 *    | (unless Elastic IP is attached)                               |
 *    +---------------------------------------------------------------+
 * 
 *   Key billing note:
 *   - RUNNING    → billed for compute (CPU/RAM) + storage (EBS)
 *   - STOPPED    → billed for storage (EBS) only, NOT compute
 *   - TERMINATED → billing stops completely (instance is gone)
 *   - ELASTIC IP → free when attached to a running instance,
 *                  charged when unattached or instance is stopped
 * 
 *    Launch
 *      │
 *      ▼
 *   [Pending] ──────────────────────────────────► [Running]
 *                  AWS provisioning (~30sec)          │
 *                                          ┌──────────┼──────────┐
 *                                          │          │          │
 *                                          ▼          ▼          ▼
 *                                      [Stopped]  [Rebooted] [Terminated]
 *                                          │          │          │
 *                                    EBS kept    Same inst   Everything
 *                                    Dynamic IP  IP stays    deleted
 *                                    released    briefly     forever ✗
 *                                          │
 *                                          ▼
 *                                      [Running]
 *                                      New dynamic IP
 * 
 * 
 * ─────────────────────────────────────────────────────────
 * 11. The Scaling Problem (Why One Instance Is Not Enough)
 * ─────────────────────────────────────────────────────────
 *
 *   - So far we have one EC2 instance running NGINX serving all
 *     incoming requests. This works fine for small traffic but
 *     creates serious problems as your application grows.
 *
 *     +---------------------------------------------------------------+
 *     |  Single EC2 — The Problem                                     |
 *     |                                                               |
 *     |  1000 users hit your server simultaneously                    |
 *     |                    |                                          |
 *     |                    V                                          |
 *     |            [EC2 Instance]                                     |
 *     |            CPU: 100%  RAM: 100%                               |
 *     |                    |                                          |
 *     |                    V                                          |
 *     |            Server crashes ✗                                   |
 *     |                                                               |
 *     |  Other problems with a single instance:                       |
 *     |  - If the instance goes down, the entire app goes down.       |
 *     |  - No fault tolerance — single point of failure.              |
 *     |  - Cannot handle traffic spikes without crashing.             |
 *     |  - Vertical scaling (adding more RAM/CPU) has a hard limit.   |
 *     +---------------------------------------------------------------+
 *
 *   Two types of scaling:
 *
 *     +------------------+------------------------------------------+
 *     |  Vertical        |  Make the single machine bigger.         |
 *     |  Scaling         |  t3.micro → t3.large → m5.xlarge         |
 *     |  (Scale Up)      |  Has a hard limit. Expensive.            |
 *     |                  |  Requires instance restart (downtime).   |
 *     +------------------+------------------------------------------+
 *     |  Horizontal      |  Add more machines of the same size.     |
 *     |  Scaling         |  1 instance → 3 instances → 10 instances |
 *     |  (Scale Out)     |  No hard limit. Cost-efficient.          |
 *     |                  |  No downtime. AWS preferred approach.    |
 *     +------------------+------------------------------------------+
 *
 *
 *    VERTICAL (Scale Up)   vs      HORIZONTAL (Scale Out)
 *
 *    Before:                       Before:
 *    ┌──────────┐                  ┌─────┐
 *    │ t3.micro │                  │ EC2 │
 *    │ 2CPU,1GB │                  └─────┘
 *    └──────────┘
 *
 *    After:                        After:
 *    ┌──────────────┐              ┌─────┐ ┌─────┐ ┌─────┐
 *    │   m5.xlarge  │              │ EC2 │ │ EC2 │ │ EC2 │
 *    │ 4CPU, 16GB   │              └─────┘ └─────┘ └─────┘
 *    └──────────────┘
 *
 *    Limit: biggest machine        Limit: none (add as many as needed)
 *    = $$$$ expensive              = cost-efficient, pay per instance
 *    Restart needed = downtime     No downtime, LB routes around it
 *
 *
 * ───────────────────────────────────────────────────────
 * 12. Load Balancer (Distributing Traffic)
 * ───────────────────────────────────────────────────────
 *
 *   What is a Load Balancer?
 *   - Imagine a restaurant with one waiter serving 100 customers.
 *     The waiter gets overwhelmed and slow. Now imagine the same
 *     restaurant hires 5 waiters and a manager who decides which
 *     waiter serves which customer. That manager is the Load Balancer.
 *   - In AWS terms: instead of one EC2 handling all requests, we run
 *     multiple EC2 instances and a Load Balancer distributes the
 *     incoming requests across all of them evenly.
 *   - Users only ever talk to the Load Balancer — they don't know
 *     (or care) which EC2 instance is actually handling their request.
 * 
 *
 *   How it works:
 *
 *                          [Internet]
 *                               │
 *                               ▼
 *                        [Load Balancer]
 *                     ← users always hit this
 *                      one DNS name, one entry point
 *                               │
 *                ┌──────────────┼──────────────┐
 *                │              │              │
 *                ▼              ▼              ▼
 *          [EC2: Inst 1]  [EC2: Inst 2]  [EC2: Inst 3]
 *           ~33% traffic   ~33% traffic   ~33% traffic
 * 
 * 
 *   What happens when one instance goes down?
 *   - The Load Balancer constantly checks if each instance is alive.
 *   - If Instance 2 goes down, it automatically stops sending
 *     traffic to it and splits traffic between the remaining ones.
 *   - The user never notices anything — the app keeps working. ✓
 *
 *                        [Load Balancer]
 *                               │
 *                ┌──────────────┘──────────────┐
 *                │                             │
 *                ▼                             ▼
 *          [EC2: Inst 1]               [EC2: Inst 3]
 *           ~50% traffic                ~50% traffic
 *           Instance 2 is down — LB reroutes automatically ✓
 * 
 * 
 *   Load Balancing Algorithms:
 *   (how the LB decides which instance gets the next request)
 *
 *     +---------------------+---------------------------------------+
 *     |  Algorithm          |  Description                          |
 *     +---------------------+---------------------------------------+
 *     |  Round Robin        |  Takes turns. Request 1 → Inst 1,     |
 *     |  (default)          |  Request 2 → Inst 2, and so on.       |
 *     |                     |  Simple and works well in most cases. |
 *     +---------------------+---------------------------------------+
 *     |  Least Connections  |  Sends the next request to whichever  |
 *     |                     |  instance is currently least busy.    |
 *     +---------------------+---------------------------------------+
 *     |  IP Hash            |  Same user always goes to the same    |
 *     |                     |  instance. Useful when the app        |
 *     |                     |  stores user session on the server.   |
 *     +---------------------+---------------------------------------+
 *     |  Weighted           |  Bigger/stronger instances get more   |
 *     |                     |  traffic than smaller ones.           |
 *     +---------------------+---------------------------------------+
 * 
 *
 *   Round Robin in action:
 *
 *    Request 1 ──────────────────────────► EC2 Instance 1
 *    Request 2 ──────────────────────────► EC2 Instance 2
 *    Request 3 ──────────────────────────► EC2 Instance 3
 *    Request 4 ──────────────────────────► EC2 Instance 1  (loops back)
 *    Request 5 ──────────────────────────► EC2 Instance 2
 *
 * 
 *   Health Checks — How the LB knows if an instance is alive:
 *   - Every 30 seconds the LB sends a small test request
 *     (GET /health) to each instance.
 *   - If the instance replies "200 OK" → it is healthy, keep sending traffic.
 *   - If it doesn't reply or replies with an error → it is unhealthy,
 *     stop sending traffic until it recovers.
 *
 *     +---------------------------------------------------------------+
 *     | [Load Balancer]  →  GET /health  →  [EC2 Instance]            |
 *     |                                                               |
 *     |  200 OK reply      → Healthy   → keeps receiving traffic      |
 *     |  No reply / error  → Unhealthy → removed from rotation        |
 *     |  Recovers later    → Healthy   → added back automatically     |
 *     |                                                               |
 *     |  - Path      : /health   (a lightweight endpoint in your app) |
 *     |  - Interval  : every 30 seconds                               |
 *     |  - Threshold : 2 failures in a row → mark unhealthy           |
 *     |                2 successes in a row → mark healthy again      |
 *     +---------------------------------------------------------------+
 * 
 * 
 *   Health check cycle:
 *
 *    Every 30 seconds:
 *    [LB] ──── GET /health ────► [EC2 #1]  →  200 OK  → ✓ healthy
 *    [LB] ──── GET /health ────► [EC2 #2]  →  200 OK  → ✓ healthy
 *    [LB] ──── GET /health ────► [EC2 #3]  →  timeout → ✗ fail #1
 *
 *    30 seconds later:
 *    [LB] ──── GET /health ────► [EC2 #3]  →  timeout → ✗ fail #2
 *    EC2 #3 marked UNHEALTHY → removed from rotation
 *    Traffic now split between EC2 #1 and EC2 #2 only
 *
 *    EC2 #3 recovers:
 *    [LB] ──── GET /health ────► [EC2 #3]  →  200 OK  → ✓ success #1
 *    [LB] ──── GET /health ────► [EC2 #3]  →  200 OK  → ✓ success #2
 *    EC2 #3 marked HEALTHY → traffic resumes ✓
 * 
 * 
 * ───────────────────────────────────────────────────────
 *   AWS Load Balancer Types
 * ───────────────────────────────────────────────────────
 *
 *   AWS offers Load Balancers as a fully managed service under
 *   ELB (Elastic Load Balancing). AWS manages everything —
 *   you just configure and use it.
 *
 *     +--------------------+------+----------------------------------+
 *     |  Type              | OSI  |  Best For                        |
 *     +--------------------+------+----------------------------------+
 *     |  Application LB    |  L7  |  Websites, APIs, microservices   |
 *     |  (ALB)             |      |  HTTP/HTTPS traffic              |
 *     +--------------------+------+----------------------------------+
 *     |  Network LB        |  L4  |  Games, VoIP, IoT devices        |
 *     |  (NLB)             |      |  TCP/UDP, needs ultra-low delay  |
 *     +--------------------+------+----------------------------------+
 *     |  Gateway LB        |  L3  |  Security tools (firewalls etc.) |
 *     |  (GWLB)            |      |  Advanced/enterprise setups      |
 *     +--------------------+------+----------------------------------+
 *
 *
 *   OSI Layer — Where ALB and NLB operate:
 *   (OSI is a model that describes how network communication works
 *    in layers. Higher layer = more information available to read.)
 *
 *    ┌──────────────────┐
 *    │ L7 Application   │ ◄── ALB reads here    NLB ignores this
 *    │ (HTTP, URL,      │     Can see: URL path, headers,
 *    │  headers,cookies)│     cookies, hostname
 *    ├──────────────────┤
 *    │ L6 Presentation  │
 *    ├──────────────────┤
 *    │ L5 Session       │
 *    ├──────────────────┤
 *    │ L4 Transport     │ ◄── Both work here     NLB stops here
 *    │ (TCP/UDP, ports) │                        Only sees IP + port
 *    ├──────────────────┤
 *    │ L3 Network       │
 *    │ (IP addresses)   │
 *    ├──────────────────┤
 *    │ L2 Data Link     │
 *    ├──────────────────┤
 *    │ L1 Physical      │
 *    └──────────────────┘
 *
 *   Simply put:
 *   - ALB is smarter — it can read the URL and decide where to route.
 *   - NLB is faster — it only looks at IP + port and forwards instantly.
 *
 * 
 *   ALB vs NLB — Key Differences:
 *
 *     +-------------------------------+---------------+---------------+
 *     |  Feature                      |  ALB (L7)     |  NLB (L4)     |
 *     +-------------------------------+---------------+---------------+
 *     |  Reads HTTP content           |  Yes          |  No           |
 *     |  Routes by URL/path/host      |  Yes          |  No           |
 *     |  Routes by IP + Port only     |  No           |  Yes          |
 *     |  Latency                      |  Milliseconds |  Microseconds |
 *     |  Static IP on LB              |  No           |  Yes          |
 *     |  Protocol                     |  HTTP/HTTPS   |  TCP/UDP/TLS  |
 *     |  Best for                     |  Web/API/SaaS |  Games/VoIP   |
 *     +-------------------------------+---------------+---------------+
 * 
 * 
 *   Which one should I use?
 *
 *    Is your app a website or API (HTTP/HTTPS)?
 *         │
 *    ┌────┴────┐
 *    │         │
 *   YES        NO
 *    │         │
 *    ▼         ▼
 *   ALB    Does it need super low latency?
 *          (e.g., game server, VoIP call, IoT device)
 *               │
 *          ┌────┴────┐
 *          │         │
 *         YES        NO
 *          │         │
 *          ▼         ▼
 *         NLB    Use ALB (safer default)
 * 
 * 
 *   When to use ALB vs NLB — quick reference:
 *
 *     +---------------------------+----------------+----------------+
 *     |  Use Case                 |  ALB           |  NLB           |
 *     +---------------------------+----------------+----------------+
 *     |  Website / REST API       |  ✓ Use this    |                |
 *     |  Microservices (routing)  |  ✓ Use this    |                |
 *     |  Route by URL path        |  ✓ Use this    |                |
 *     |  Handle SSL (HTTPS)       |  ✓ Use this    |                |
 *     |  Online game server       |                |  ✓ Use this    |
 *     |  Video / voice calls      |                |  ✓ Use this    |
 *     |  Need a fixed IP on LB    |                |  ✓ Use this    |
 *     |  Non-HTTP protocol        |                |  ✓ Use this    |
 *     |  Handle millions req/sec  |                |  ✓ Use this    |
 *     +---------------------------+----------------+----------------+
 * 
 * 
 * ───────────────────────────────────────────────────────
 *   Type 1 — ALB: Application Load Balancer (Layer 7)
 * ───────────────────────────────────────────────────────
 *
 *   - ALB works at Layer 7 — meaning it can actually read and
 *     understand the HTTP request coming from the user.
 *   - It can see: what URL was requested, what hostname was used,
 *     what headers were sent, and whether it is HTTP or HTTPS.
 *   - Using this information, it makes smart routing decisions.
 *
 *
 *   ALB Feature 1 — Path-Based Routing:
 *   Route different URL paths to different sets of servers.
 *
 *   Real world example: chaicode.com has one domain but many services.
 *   Instead of building everything on one big server, each service
 *   runs on its own EC2 instances. ALB routes to the right one.
 *
 *                    [ALB]
 *                      │
 *         ┌────────────┼─────────────┐
 *         │            │             │
 *    /api/users   /api/orders   /api/payments
 *         │            │             │
 *         ▼            ▼             ▼
 *    ┌─────────┐  ┌─────────┐  ┌──────────┐
 *    │ Users   │  │ Orders  │  │ Payments │
 *    │ Service │  │ Service │  │ Service  │
 *    │ (2 EC2) │  │ (3 EC2) │  │ (2 EC2)  │
 *    └─────────┘  └─────────┘  └──────────┘
 *
 *   Orders service getting more traffic? → scale Orders only.
 *   No need to scale Users or Payments. Saves money. ✓
 *
 *
 *   ALB Feature 2 — Host-Based Routing:
 *   Route different subdomains to different servers.
 *
 *    api.chaicode.com     → goes to API servers
 *    admin.chaicode.com   → goes to Admin servers
 *    static.chaicode.com  → goes to Static file servers
 *
 *   All of this is handled by one ALB — no need for separate
 *   load balancers per subdomain.
 *
 *
 *   ALB Feature 3 — SSL Termination:
 *   ALB handles HTTPS encryption so your EC2 doesn't have to.
 *
 *   Without SSL termination:
 *   - Every EC2 instance needs an SSL certificate installed.
 *   - Every EC2 instance wastes CPU decrypting HTTPS traffic.
 *   - Managing SSL renewal on 10 servers is painful.
 *
 *   With SSL termination at ALB:
 *   - SSL certificate lives only on the ALB — one place.
 *   - EC2 instances only receive plain HTTP — no decryption needed.
 *   - CPU on EC2 is freed up for your actual application.
 *
 *    Internet (encrypted HTTPS)    Inside AWS (plain HTTP)
 *    ┌───────────────┐             ┌───────────────────────┐
 *    │  User Browser │             │   EC2 Instances       │
 *    │               │             │  ┌──────┐ ┌──────┐    │
 *    │  HTTPS ───────┼────────────►│  │ HTTP │ │ HTTP │    │
 *    │  (encrypted)  │    [ALB]    │  └──────┘ └──────┘    │
 *    │               │  decrypts   │  simple, no SSL ✓     │
 *    └───────────────┘  here       └───────────────────────┘
 *
 *
 *   ALB — All Features at a glance:
 *
 *     +------------------------------+--------------------------------+
 *     |  Feature                     |  What it does                  |
 *     +------------------------------+--------------------------------+
 *     |  Path-based routing          |  /users → Users EC2            |
 *     |                              |  /orders → Orders EC2          |
 *     +------------------------------+--------------------------------+
 *     |  Host-based routing          |  api.site.com → API servers    |
 *     |                              |  admin.site.com → Admin servers|
 *     +------------------------------+--------------------------------+
 *     |  SSL/TLS Termination         |  Handles HTTPS for you         |
 *     |                              |  EC2 only gets plain HTTP      |
 *     +------------------------------+--------------------------------+
 *     |  Health Checks               |  Removes broken instances      |
 *     |                              |  automatically                 |
 *     +------------------------------+--------------------------------+
 *     |  Sticky Sessions             |  Same user always hits same    |
 *     |                              |  EC2 (useful for sessions)     |
 *     +------------------------------+--------------------------------+
 *     |  WebSocket Support           |  Works for real-time apps      |
 *     |                              |  like chat or live updates     |
 *     +------------------------------+--------------------------------+
 *     |  Works with ECS / Lambda     |  Not just EC2 — works with     |
 *     |                              |  containers and serverless too |
 *     +------------------------------+--------------------------------+
 *
 *
 *   How to Create an ALB (step by step):
 *
 *   PART A — Create a Target Group:
 *   (A Target Group is simply the list of EC2 instances the ALB
 *    should send traffic to. Think of it as a group of waiters.)
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Target Groups"
 *       → Click "Create Target Group"
 *       → Target type      : Instances
 *       → Protocol         : HTTP
 *       → Port             : 80
 *       → Health check path: /health
 *       → Click "Next"
 *       → Select your EC2 instances → "Include as pending"
 *       → Click "Create Target Group"
 *
 *   PART B — Create the ALB:
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Load Balancers"
 *       → Click "Create Load Balancer"
 *       → Select: "Application Load Balancer"
 *       → Name              : chaicode-alb
 *       → Scheme            : Internet-facing
 *           (Internet-facing = public, accessible from internet)
 *           (Internal = private, only accessible inside AWS network)
 *       → IP address type   : IPv4
 *       → VPC               : select your VPC
 *       → Availability Zones: select at least 2 AZs
 *           (always pick 2+ AZs — if one AZ goes down, ALB stays up)
 *       → Security Group    : LB security group (allow port 80/443)
 *       → Listener          : HTTP port 80 → forward to Target Group
 *       → Click "Create Load Balancer"
 *
 *   PART C — Lock down EC2 so only ALB can reach it:
 *   (Right now anyone on the internet can directly hit your EC2
 *    on port 80, bypassing the LB entirely. We must fix this.)
 *
 *     BEFORE (bad — EC2 open to everyone):
 *     +----------+------+------------+----------------------------+
 *     |  HTTP    |  80  | 0.0.0.0/0  |  Anyone on internet ✗     |
 *     +----------+------+------------+----------------------------+
 *
 *     AFTER (good — only ALB can reach EC2):
 *     +----------+------+---------------------+------------------+
 *     |  HTTP    |  80  | ALB Security Group  |  Only ALB ✓      |
 *     +----------+------+---------------------+------------------+
 *
 *
 *   Why this matters:
 *
 *    BAD:  User ──► EC2 directly   ← bypasses LB, unbalanced ✗
 *    BAD:  User ──► EC2 directly   ← attacker can hit EC2 directly ✗
 *
 *    GOOD: User ──► ALB ──► EC2   ← all traffic goes through LB ✓
 *          User ──► EC2 directly  ← BLOCKED by Security Group ✓
 *
 *
 *   PART D — Test the ALB:
 *
 *     EC2 Dashboard → Load Balancers
 *       → Copy the ALB "DNS Name"
 *           e.g., chaicode-alb-123456.ap-south-1.elb.amazonaws.com
 *       → Paste in browser → your app should load ✓
 *       → Refresh multiple times → each refresh may hit a different EC2
 *
 *   PART E — Point GoDaddy Domain to the ALB:
 *
 *   - ALB does not have a fixed IP address — it only has a DNS name.
 *   - Because of this we cannot use an A record (A records need an IP).
 *   - Instead we use a CNAME record which maps one DNS name to another.
 *
 *     +--------+---------+------------------------------------------+
 *     |  Type  |  Name   |  Value                                   |
 *     +--------+---------+------------------------------------------+
 *     |  CNAME |  www    |  chaicode-alb-123456.ap-south-1.elb.     |
 *     |        |         |  amazonaws.com                           |
 *     +--------+---------+------------------------------------------+
 *
 *   Note: GoDaddy does not allow CNAME on the root domain (@).
 *   For the root domain (chaicode.com without www) →
 *   use AWS Route 53 which supports Alias records for this purpose.
 *
 *
 *   How to Delete an ALB:
 *
 *     EC2 Dashboard → Load Balancers
 *       → Select the ALB
 *       → Click "Actions" → "Delete Load Balancer"
 *       → Confirm → Click "Delete"
 *
 *     What happens after:
 *     - ALB DNS name stops working immediately.
 *     - Target Group is NOT deleted — delete it separately.
 *     - EC2 instances keep running, they are unaffected.
 *     - ALB billing stops immediately.
 *
 *     Clean up in this order:
 *     1. Delete Load Balancer
 *     2. Delete Target Group
 *     3. Revert EC2 Security Group rules if needed
 *
 *
 * ───────────────────────────────────────────────────────
 *   Type 2 — NLB: Network Load Balancer (Layer 4)
 * ───────────────────────────────────────────────────────
 *
 *   - NLB works at Layer 4 — it can only see the IP address and
 *     port number of the incoming packet. It cannot read the URL,
 *     headers, or any HTTP content at all.
 *   - Because it does less work per request, it is extremely fast —
 *     microsecond latency vs milliseconds for ALB.
 *   - Best for workloads where speed matters more than smart routing:
 *     gaming servers, VoIP calls, IoT devices, and so on.
 *
 *
 *   How NLB routes traffic:
 *   (it only sees IP + port — nothing else)
 *
 *    [Client sends a TCP packet]
 *        │
 *        │  NLB sees: destination port 443, source IP 103.x.x.x
 *        │  NLB does NOT see: URL, headers, cookies, any HTTP data
 *        ▼
 *    [NLB — forwards to least loaded target]
 *        │
 *        ┌──────────────┬──────────────┐
 *        │              │              │
 *        ▼              ▼              ▼
 *   [EC2: Inst 1]  [EC2: Inst 2]  [EC2: Inst 3]
 *   packet arrives  packet arrives  packet arrives
 *   instantly ✓     instantly ✓     instantly ✓
 *
 *
 *   How to Create an NLB (step by step):
 *
 *   PART A — Create a Target Group:
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Target Groups"
 *       → Click "Create Target Group"
 *       → Target type      : Instances
 *       → Protocol         : TCP
 *           (NLB uses TCP/UDP — not HTTP like ALB)
 *       → Port             : 80
 *       → Health check     : TCP (or HTTP /health)
 *       → Select EC2 instances → "Include as pending"
 *       → Click "Create Target Group"
 *
 *   PART B — Create the NLB:
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Load Balancers"
 *       → Click "Create Load Balancer"
 *       → Select: "Network Load Balancer"
 *       → Name              : chaicode-nlb
 *       → Scheme            : Internet-facing
 *       → IP address type   : IPv4
 *       → VPC               : select your VPC
 *       → Availability Zones: select at least 2 AZs
 *           → Optionally assign an Elastic IP per AZ here
 *             (unlike ALB, NLB supports a fixed static IP ✓)
 *       → Listener          : TCP port 80 → forward to Target Group
 *       → Click "Create Load Balancer"
 *
 *   PART C — Point GoDaddy Domain to the NLB:
 *
 *   - Since NLB supports Elastic IP, you can use a simple A record.
 *
 *     Option A — Use Elastic IP (static, most reliable):
 *     +--------+---------+-----------------+
 *     |  Type  |  Name   |  Value          |
 *     +--------+---------+-----------------+
 *     |  A     |  @      |  54.23.11.5     |  ← NLB Elastic IP
 *     +--------+---------+-----------------+
 *
 *     Option B — Use DNS name (CNAME, no Elastic IP assigned):
 *     +--------+---------+------------------------------------------+
 *     |  Type  |  Name   |  Value                                   |
 *     +--------+---------+------------------------------------------+
 *     |  CNAME |  www    |  chaicode-nlb-123456.ap-south-1.elb.     |
 *     |        |         |  amazonaws.com                           |
 *     +--------+---------+------------------------------------------+
 *
 *   PART D — Test the NLB:
 *
 *     EC2 Dashboard → Load Balancers
 *       → Copy the NLB "DNS Name" or use the Elastic IP
 *       → Test from browser or terminal:
 *           curl http://<NLB-DNS-or-IP>
 *       → Traffic should reach your EC2 instances ✓
 *
 *
 *   How to Delete an NLB:
 *
 *     EC2 Dashboard → Load Balancers
 *       → Select the NLB
 *       → Click "Actions" → "Delete Load Balancer"
 *       → Confirm → Click "Delete"
 *
 *     What happens after:
 *     - NLB DNS name stops working immediately.
 *     - If an Elastic IP was assigned, it becomes unattached →
 *       release it right away or AWS will start charging you.
 *     - Target Group must be deleted separately.
 *     - EC2 instances are not affected.
 *
 *     Clean up in this order:
 *     1. Delete Load Balancer
 *     2. Release Elastic IP (if one was assigned)
 *     3. Delete Target Group
 *
 *
 * ───────────────────────────────────────────────────────
 * 13. Auto Scaling (Automatic Instance Management)
 * ───────────────────────────────────────────────────────
 *
 *   The Problem with Manual Scaling:
 *   - Even with a Load Balancer, we are still manually deciding
 *     how many EC2 instances to keep running at all times.
 *   - But traffic is unpredictable — it goes up and down throughout
 *     the day. If we keep too few instances the app crashes under
 *     load. If we keep too many we waste money on idle servers.
 *
 *
 *   Traffic pattern vs manual scaling problem:
 *
 *    Traffic (requests/sec)
 *    │
 *    │                    ▲ spike (lunch hour)
 *    │                   ╱│╲
 *    │                  ╱ │ ╲
 *    │                 ╱  │  ╲
 *    │________________╱   │   ╲________________
 *    └─────────────────────────────────────────► time
 *    12am  3am  6am  9am 12pm  3pm  6pm  9pm
 *
 *    With fixed 3 instances:
 *    - 3am:  3 instances sitting idle   → wasting money ✗
 *    - 12pm: 3 instances all maxed out  → app crashes ✗
 *
 *    With Auto Scaling:
 *    - 3am:  auto scales down to 2  → saves money ✓
 *    - 12pm: auto scales up to 8    → handles load ✓
 *    - 3pm:  auto scales back to 3  → back to normal ✓
 *
 *
 *   The Solution — Auto Scaling Group (ASG):
 *   - An Auto Scaling Group is a feature that watches your EC2
 *     instances and automatically adds more when traffic goes up,
 *     and removes them when traffic goes down.
 *   - You just set the rules once — AWS does the rest.
 *   - New instances are automatically added to the Load Balancer
 *     when launched, and removed when terminated. No manual steps.
 *
 *     +---------------------------------------------------------------+
 *     |  You define three numbers:                                    |
 *     |                                                               |
 *     |  Minimum : 2  → never go below 2 instances (always on)        |
 *     |  Desired : 3  → keep 3 running under normal traffic           |
 *     |  Maximum : 10 → never go above 10 (cost protection)           |
 *     |                                                               |
 *     |  [Low Traffic — 3 AM]                                         |
 *     |  Only 2 instances running (minimum) → saving money ✓          |
 *     |                                                               |
 *     |  [Traffic spike — 12 PM]                                      |
 *     |  CPU > 70% → ASG adds 2 more instances automatically ✓        |
 *     |  New instances registered with ALB → traffic balanced ✓       |
 *     |                                                               |
 *     |  [Traffic drops — 2 AM]                                       |
 *     |  CPU < 30% → ASG removes extra instances automatically ✓      |
 *     +---------------------------------------------------------------+
 *
 *
 *   ASG min/desired/max boundaries:
 *
 *    Instance count
 *    │
 *  10│─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─  MAX (hard ceiling — cost cap)
 *    │
 *   7│                    ╭──────╮
 *   6│                   ╱        ╲
 *   5│                  ╱          ╲     ASG scales up during
 *   4│                 ╱            ╲    peak hours and back
 *   3│────────────────╯              ╲──────────  down after
 *    │
 *   2│─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─  MIN (always kept running)
 *   1│
 *    └─────────────────────────────────────────────► time
 *
 *
 *   Auto Scaling Policies — what triggers scaling:
 *
 *     +----------------------+----------------------------------------+
 *     |  Policy              |  How it works                          |
 *     +----------------------+----------------------------------------+
 *     |  Target Tracking     |  You say "keep CPU at 50%".            |
 *     |  (recommended)       |  AWS figures out how many instances    |
 *     |                      |  are needed to maintain that target.   |
 *     +----------------------+----------------------------------------+
 *     |  Step Scaling        |  You define thresholds manually:       |
 *     |                      |  CPU > 70% → add 2 instances           |
 *     |                      |  CPU > 90% → add 5 instances           |
 *     |                      |  CPU < 30% → remove 1 instance         |
 *     +----------------------+----------------------------------------+
 *     |  Simple Scaling      |  A single CloudWatch alarm triggers    |
 *     |                      |  one fixed add/remove action.          |
 *     |                      |  Waits for cooldown before acting again.|
 *     +----------------------+----------------------------------------+
 *     |  Scheduled Scaling   |  Scale at a fixed time.                |
 *     |                      |  e.g., every Monday 9 AM add 3         |
 *     |                      |  instances, every Friday 9 PM remove 3.|
 *     +----------------------+----------------------------------------+
 *     |  Predictive Scaling  |  AWS uses machine learning to predict  |
 *     |                      |  traffic and scales BEFORE the spike   |
 *     |                      |  arrives. Great for known patterns.    |
 *     +----------------------+----------------------------------------+
 *
 *
 *   Step Scaling example:
 *
 *    CPU%  │  What ASG does
 *    ──────┼─────────────────────────────────────────
 *    < 30% │  Remove 1 instance (save cost)
 *    30-70%│  Do nothing (everything is fine)
 *    > 70% │  Add 2 more instances
 *    > 90% │  Add 5 more instances immediately
 *
 *    Real scenario:
 *    CPU 85% → add 2 instances → CPU drops to 55% → stable ✓
 *    CPU 95% → add 5 instances → CPU drops to 40% → stable ✓
 *    Traffic dies → CPU 15% → remove instances one by one ✓
 *
 *
 *   Launch Template — the blueprint for new instances:
 *   - When ASG decides to add a new instance, it needs to know
 *     exactly what kind of instance to create.
 *   - A Launch Template is like a saved recipe — it contains all
 *     the settings for a new EC2 instance.
 *   - Every new instance ASG creates uses this exact template.
 *     No manual setup needed for each new server.
 *
 *     What goes inside a Launch Template:
 *     +---------------------------------------------------------------+
 *     |  AMI ID          :  Ubuntu 22.04  (which OS to install)       |
 *     |  Instance Type   :  t3.medium     (how much CPU/RAM)          |
 *     |  Key Pair        :  my-key.pem    (for SSH access)            |
 *     |  Security Group  :  sg-xxxxxxxx   (firewall rules)            |
 *     |  IAM Role        :  EC2-S3-Role   (AWS permissions)           |
 *     |  User Data Script:  #!/bin/bash                               |
 *     |                     sudo apt update                           |
 *     |                     sudo apt install nginx -y                 |
 *     |                     sudo systemctl start nginx                |
 *     |                     ← this script runs automatically          |
 *     |                       on every new instance at boot           |
 *     +---------------------------------------------------------------+
 *
 *
 *   Launch Template + ASG in action:
 *
 *    CPU spikes → ASG decides: "I need 2 more instances"
 *                                    │
 *                                    ▼
 *                          [Launch Template]
 *                          OS: Ubuntu 22.04
 *                          Type: t3.medium
 *                          Script: install + start nginx
 *                                    │
 *                          ┌─────────┴─────────┐
 *                          │                   │
 *                          ▼                   ▼
 *                    [New EC2 #4]         [New EC2 #5]
 *                    boots Ubuntu          boots Ubuntu
 *                    runs script           runs script
 *                    nginx starts ✓        nginx starts ✓
 *                          │                   │
 *                          └─────────┬─────────┘
 *                                    ▼
 *                          [ALB Target Group]
 *                          Both registered automatically ✓
 *                          Traffic now spread across 5 instances
 *
 *
 *   How to Create an Auto Scaling Group (step by step):
 *
 *   PART A — Create a Launch Template:
 *   (Save a template so ASG knows what to launch)
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Launch Templates"
 *       → Click "Create Launch Template"
 *       → Name             : chaicode-template
 *       → AMI              : Ubuntu 22.04 LTS
 *       → Instance Type    : t3.medium
 *       → Key Pair         : my-key.pem
 *       → Security Group   : existing EC2 security group
 *       → Advanced Details → User Data:
 *           #!/bin/bash
 *           sudo apt update -y
 *           sudo apt install nginx -y
 *           sudo systemctl start nginx
 *           sudo systemctl enable nginx
 *       → Click "Create Launch Template"
 *
 *   PART B — Create the Auto Scaling Group:
 *
 *     EC2 Dashboard
 *       → Left Sidebar: "Auto Scaling Groups"
 *       → Click "Create Auto Scaling Group"
 *       → Name              : chaicode-asg
 *       → Launch Template   : chaicode-template
 *       → VPC               : select your VPC
 *       → Availability Zones: select at least 2 AZs
 *           (spread instances across AZs — if one AZ goes down,
 *            instances in the other AZ keep your app running)
 *       → Attach to Load Balancer: select your ALB Target Group
 *       → Health Check Type : ELB
 *           (LB health checks are more reliable than EC2 checks alone)
 *       → Group Size:
 *           Minimum capacity : 2
 *           Desired capacity : 3
 *           Maximum capacity : 10
 *       → Scaling Policy    : Target Tracking
 *           Metric           : Average CPU Utilization
 *           Target value     : 50%
 *           (ASG will add/remove instances to keep average CPU ~50%)
 *       → Click "Create Auto Scaling Group"
 *
 *   PART C — Verify it is working:
 *
 *     EC2 Dashboard → Auto Scaling Groups → select yours
 *       → "Activity" tab   → shows every scale up/down event
 *       → "Instances" tab  → shows all currently running instances
 *       → "Monitoring" tab → shows CPU graph and scaling triggers
 *
 *
 *   Cooldown Period — preventing over-reaction:
 *   - After ASG adds or removes instances, it waits 5 minutes
 *     (300 seconds) before taking any further action.
 *   - Why? Because new instances take time to boot and start
 *     receiving traffic. Without cooldown, ASG might keep adding
 *     instances even though the previous ones haven't even started yet.
 *
 *
 *   Cooldown prevents chaos:
 *
 *    WITHOUT cooldown (bad):
 *    CPU 80% → add 2 instances
 *    CPU 75% → add 2 more (previous ones not ready yet!) ✗
 *    CPU 50% → remove 4 (now we have too few!) ✗
 *    CPU 85% → add 4 again (back to chaos) ✗
 *
 *    WITH cooldown (good):
 *    CPU 80% → add 2 instances
 *              wait 300 seconds for them to boot and stabilise
 *    CPU 52% → everything is fine, no action needed ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * FINAL ARCHITECTURE — EC2 + Load Balancer + Auto Scaling
 * ───────────────────────────────────────────────────────
 *
 *     +---------------------------------------------------------------+
 *     |  [User] types chaicode.com                                    |
 *     |          |                                                    |
 *     |          V                                                    |
 *     |  [GoDaddy DNS]                                                |
 *     |  CNAME: chaicode.com → ALB DNS name                           |
 *     |          |                                                    |
 *     |          V                                                    |
 *     |  [ALB — Application Load Balancer]                            |
 *     |  - Receives all traffic                                       |
 *     |  - Handles SSL (HTTPS)                                        |
 *     |  - Routes by path or host                                     |
 *     |  - Health checks all instances                                |
 *     |          |                                                    |
 *     |    +-----+------+------+------+                               |
 *     |    |            |      |      |                               |
 *     |    V            V      V      V                               |
 *     | [EC2:1]     [EC2:2] [EC2:3] [EC2:4]  ← Auto Scaling Group     |
 *     |  NGINX       NGINX   NGINX   NGINX      adds/removes          |
 *     |  (AZ-1)      (AZ-1)  (AZ-2)  (AZ-2)    based on traffic       |
 *     |    |            |      |      |                               |
 *     |    +-----+------+------+------+                               |
 *     |          |                                                    |
 *     |          V                                                    |
 *     |     [Database — RDS / DynamoDB]                               |
 *     |                                                               |
 *     |  ASG: Min:2 | Desired:3 | Max:10                              |
 *     |  CPU > 70% → add instance | CPU < 30% → remove instance       |
 *     +---------------------------------------------------------------+
 *
 *
 *   Full system — all layers labelled:
 *
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │                    INTERNET                                 │
 *    └────────────────────────┬────────────────────────────────────┘
 *                             │  User types chaicode.com
 *                             ▼
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  GoDaddy DNS                                                │
 *    │  CNAME: chaicode.com → chaicode-alb-xxx.elb.amazonaws.com   │
 *    └────────────────────────┬────────────────────────────────────┘
 *                             │
 *                             ▼
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  ALB  (Application Load Balancer)                           │
 *    │  Handles SSL | Routes by path/host | Runs health checks     │
 *    └────────┬───────────────┬──────────────┬──────────────┬──────┘
 *             │               │              │              │
 *             ▼               ▼              ▼              ▼
 *    ┌────────────┐  ┌────────────┐  ┌────────────┐  ┌───────────┐
 *    │  EC2 #1    │  │  EC2 #2    │  │  EC2 #3    │  │  EC2 #4   │
 *    │  AZ: 1a    │  │  AZ: 1a    │  │  AZ: 1b    │  │  AZ: 1b   │
 *    │  NGINX     │  │  NGINX     │  │  NGINX     │  │  NGINX    │
 *    │  EBS: 8GB  │  │  EBS: 8GB  │  │  EBS: 8GB  │  │  EBS: 8GB │
 *    │  SG: ALB   │  │  SG: ALB   │  │  SG: ALB   │  │  SG: ALB  │
 *    │  only      │  │  only      │  │  only      │  │  only     │
 *    └────────────┘  └────────────┘  └────────────┘  └───────────┘
 *         ▲               ▲                               ▲
 *         └───────────────┴───── Auto Scaling Group ──────┘
 *                                Min:2  Desired:3  Max:10
 *                                CPU > 70% → add instance
 *                                CPU < 30% → remove instance
 *                                Uses Launch Template to boot new EC2s
 *             │
 *             ▼
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  Database Layer                                             │
 *    │  RDS (MySQL/Postgres) or DynamoDB                           │
 *    │  Accessible from EC2 Security Group only — not internet     │
 *    └─────────────────────────────────────────────────────────────┘
 *
 *
 *   Summary — what each layer does:
 *
 *     +------------------+------------------------------------------+
 *     |  Component       |  Responsibility                           |
 *     +------------------+------------------------------------------+
 *     |  GoDaddy DNS     |  Translates domain name to ALB address   |
 *     +------------------+------------------------------------------+
 *     |  Elastic IP      |  Fixed IP for direct EC2 access (dev)    |
 *     +------------------+------------------------------------------+
 *     |  ALB             |  Single entry, distributes traffic, SSL  |
 *     +------------------+------------------------------------------+
 *     |  EC2 Instances   |  Actually runs your application          |
 *     +------------------+------------------------------------------+
 *     |  Security Group  |  Firewall — controls who reaches EC2     |
 *     +------------------+------------------------------------------+
 *     |  EBS Volume      |  Hard disk — persistent data storage     |
 *     +------------------+------------------------------------------+
 *     |  Auto Scaling    |  Adds/removes EC2s based on traffic      |
 *     |  Group (ASG)     |  — saves cost, prevents crashes          |
 *     +------------------+------------------------------------------+
 *     |  Launch Template |  Saved blueprint ASG uses to create      |
 *     |                  |  new EC2 instances automatically          |
 *     +------------------+------------------------------------------+
 */


In [ ]:

/**
 * ───────────────────────────────────────────────────────
 * From EC2 to ECS — Why We Need Containers
 * ───────────────────────────────────────────────────────
 *
 * ───────────────────────────────────────────────────────
 * PART 1 — The Problems with Raw EC2
 * ───────────────────────────────────────────────────────
 *
 *   We learned how to run applications on EC2 instances.
 *   For simple apps this works fine. But as your app grows
 *   and your team grows, EC2 alone starts showing problems.
 *
 *
 *   Problem 1 — "It works on my machine"
 *
 *   - Developer builds the app on their laptop. Works perfectly.
 *   - They deploy it to EC2. It crashes.
 *   - Reason: different Node.js version, different library versions,
 *     different OS settings between laptop and EC2.
 *
 *    Your Laptop             Teammate's Laptop        EC2 Server
 *    ┌──────────────┐        ┌──────────────┐         ┌──────────────┐
 *    │ Node.js v18  │        │ Node.js v14  │         │ Node.js v16  │
 *    │ npm v9       │        │ npm v6       │         │ npm v8       │
 *    │ macOS        │        │ Windows      │         │ Ubuntu       │
 *    │              │        │              │         │              │
 *    │  App ✓       │  ────► │  App ✗      │  ────►  │  App ✗       │
 *    │  works here  │        │  crashes     │         │  crashes     │
 *    └──────────────┘        └──────────────┘         └──────────────┘
 *
 *    Every developer has a slightly different environment.
 *    EC2 has its own environment. Nothing matches.
 *
 *
 *   Problem 2 — Deploying updates is painful
 *
 *   - You have 10 EC2 instances running your app.
 *   - You release a new version. Now you must:
 *     1. SSH into each instance one by one
 *     2. Pull the latest code
 *     3. Install dependencies
 *     4. Restart the server
 *     5. Hope nothing breaks mid-deploy
 *   - If step 3 fails on instance 6, instances 1-5 run new code
 *     and 6-10 still run old code — your app is inconsistent.
 *
 *    Deploying v2 to 10 EC2 instances manually:
 *
 *    EC2 #1  ──► ssh → git pull → npm install → restart  ✓ v2
 *    EC2 #2  ──► ssh → git pull → npm install → restart  ✓ v2
 *    EC2 #3  ──► ssh → git pull → npm install → restart  ✓ v2
 *    EC2 #4  ──► ssh → git pull → npm install → restart  ✓ v2
 *    EC2 #5  ──► ssh → git pull → npm install → restart  ✓ v2
 *    EC2 #6  ──► ssh → git pull → npm install → FAILS ✗  still v1
 *    EC2 #7-10 ──► not even attempted                     still v1
 *
 *    Result: half the servers on v1, half on v2 — inconsistent ✗
 *
 *
 *   Problem 3 — Wasted resources
 *
 *   - Each EC2 instance runs one app and is often 70-80% idle.
 *   - You are paying for CPU and RAM that is not being used.
 *
 *    EC2 instance (t3.large — 2 CPU, 8 GB RAM):
 *    ┌─────────────────────────────────────┐
 *    │  App: Users Service                 │
 *    │  Uses: 0.3 CPU, 1 GB RAM            │
 *    │                                     │
 *    │  Idle: 1.7 CPU, 7 GB RAM  ← wasted  │
 *    └─────────────────────────────────────┘
 *    Paying for 8 GB but only using 1 GB.
 *
 *
 *   Problem 4 — Cannot scale individual services
 *
 *   - Suppose your app has 3 services: Users, Orders, Payments.
 *   - All 3 run on the same EC2 instance.
 *   - Orders gets a traffic spike.
 *   - You cannot scale just Orders — you must scale the entire
 *     EC2 instance, which also scales Users and Payments for no reason.
 *
 *    ┌──────────────────────────────────────┐
 *    │           EC2 Instance               │
 *    │  ┌────────┐ ┌────────┐ ┌──────────┐  │
 *    │  │ Users  │ │ Orders │ │ Payments │  │
 *    │  │ 10 req │ │ 500 req│ │ 10 req   │  │
 *    │  │ /sec   │ │ /sec ↑ │ │ /sec     │  │
 *    │  └────────┘ └────────┘ └──────────┘  │
 *    └──────────────────────────────────────┘
 *    Orders needs more capacity →
 *    scale entire EC2 → Users and Payments waste the extra capacity ✗
 *
 *
 *   Problem 5 — Dependency conflicts between services
 *
 *   - Users service needs Python 3.9.
 *   - ML service needs Python 3.6 (old library compatibility).
 *   - You cannot install both cleanly on the same EC2 instance.
 *
 *    ┌──────────────────────────────────────┐
 *    │           EC2 Instance               │
 *    │  Users Service → needs Python 3.9    │
 *    │  ML Service    → needs Python 3.6    │
 *    │                                      │
 *    │  Which Python do we install? ✗       │
 *    │  They conflict on the same machine.  │
 *    └──────────────────────────────────────┘
 *
 *
 *   Summary — EC2 problems at scale:
 *
 *     +-------------------------------+--------------------------------+
 *     |  Problem                      |  Impact                        |
 *     +-------------------------------+--------------------------------+
 *     |  Different environments       |  App works on laptop not EC2   |
 *     |  Manual deploy to each EC2    |  Slow, inconsistent, risky     |
 *     |  Wasted idle resources        |  Paying for unused CPU/RAM     |
 *     |  Cannot scale one service     |  Scale all-or-nothing          |
 *     |  Dependency conflicts         |  Services break each other     |
 *     +-------------------------------+--------------------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * PART 2 — The Solution: Containers
 * ───────────────────────────────────────────────────────
 *
 *   What is a Container?
 *   - A Container is a lightweight, portable box that packages
 *     your application together with everything it needs to run:
 *     code, runtime, libraries, dependencies, and config.
 *   - This box runs identically on any machine — your laptop,
 *     your teammate's machine, or any EC2 server.
 *   - "If it runs in the container on your machine, it runs
 *     everywhere." That is the core promise of containers.
 *
 *
 *   Container analogy — physical shipping containers:
 *
 *    Physical world:
 *    A shipping container can be loaded on a truck, a ship, or a
 *    train. The contents don't change. The container fits everywhere.
 *
 *    Software world:
 *    A software container runs on your laptop, EC2, or any cloud.
 *    The app inside doesn't change. It behaves identically everywhere.
 *
 *
 *   Container vs Virtual Machine (EC2):
 *
 *    Virtual Machine (EC2):              Container:
 *    ┌────────────────────┐              ┌────────────────────┐
 *    │   Your App         │              │   Your App         │
 *    ├────────────────────┤              ├────────────────────┤
 *    │   Full Guest OS    │              │   App Libraries    │
 *    │   (Ubuntu, Windows)│              │   (no full OS)     │
 *    ├────────────────────┤              ├────────────────────┤
 *    │   Hypervisor       │              │   Container Engine │
 *    ├────────────────────┤              │   (Docker)         │
 *    │   Physical Server  │              ├────────────────────┤
 *    └────────────────────┘              │   Host OS          │
 *                                        ├────────────────────┤
 *                                        │   Physical Server  │
 *                                        └────────────────────┘
 *
 *     +------------------+--------------------+--------------------+
 *     |  Feature         |  VM (EC2)          |  Container         |
 *     +------------------+--------------------+--------------------+
 *     |  Includes OS     |  Yes (full OS)     |  No (shares host)  |
 *     |  Size            |  GBs               |  MBs               |
 *     |  Startup time    |  Minutes           |  Seconds           |
 *     |  Isolation       |  Strong            |  Good              |
 *     |  Resource usage  |  Heavy             |  Lightweight       |
 *     |  Portability     |  Limited           |  Runs anywhere     |
 *     +------------------+--------------------+--------------------+
 *
 *
 *   How containers solve every EC2 problem:
 *
 *     +-------------------------------+--------------------------------+
 *     |  EC2 Problem                  |  Container Solution            |
 *     +-------------------------------+--------------------------------+
 *     |  Different environments       |  Same container runs anywhere  |
 *     |  Manual deploy to each EC2    |  Deploy a new image — done ✓   |
 *     |  Wasted idle resources        |  Pack multiple containers per  |
 *     |                               |  machine — share resources ✓  |
 *     |  Cannot scale one service     |  Each service = one container  |
 *     |                               |  Scale containers independently|
 *     |  Dependency conflicts         |  Each container is isolated ✓  |
 *     |                               |  Python 3.9 and 3.6 coexist   |
 *     +-------------------------------+--------------------------------+
 *
 *
 *   Multiple containers on one EC2 instance:
 *
 *    ┌──────────────────────────────────────────────────────┐
 *    │  EC2 Instance (t3.large — 2 CPU, 8 GB RAM)           │
 *    │  Resources fully utilised ✓                          │
 *    │                                                      │
 *    │  ┌────────────┐  ┌────────────┐  ┌────────────┐      │
 *    │  │ Container 1│  │ Container 2│  │ Container 3│      │
 *    │  │ Users Svc  │  │ Orders Svc │  │ ML Service │      │
 *    │  │ Node v18   │  │ Node v18   │  │ Python 3.9 │      │
 *    │  │ 0.5 CPU    │  │ 1.0 CPU    │  │ 0.3 CPU    │      │
 *    │  │ 1 GB RAM   │  │ 2 GB RAM   │  │ 1 GB RAM   │      │
 *    │  └────────────┘  └────────────┘  └────────────┘      │
 *    │                                                      │
 *    │  Each container is isolated — no conflicts ✓         │
 *    │  Resources packed efficiently — less waste ✓         │
 *    └──────────────────────────────────────────────────────┘
 *
 *
 * ───────────────────────────────────────────────────────
 * PART 3 — What is Docker?
 * ───────────────────────────────────────────────────────
 *
 *   - Docker is the most popular tool for creating and running containers.
 *   - Docker is to containers what AWS is to cloud servers.
 *   - You write a Dockerfile (a recipe), Docker builds an Image
 *     from it, and you run that Image as a Container.
 *
 *
 *   Three key Docker terms:
 *
 *     +------------------+------------------------------------------+
 *     |  Term            |  What it is                              |
 *     +------------------+------------------------------------------+
 *     |  Dockerfile      |  A text file with step-by-step           |
 *     |                  |  instructions for building your image.   |
 *     |                  |  Recipe: install Node, copy code,        |
 *     |                  |  run npm install, expose port, start app.|
 *     +------------------+------------------------------------------+
 *     |  Image           |  The built, packaged snapshot created    |
 *     |                  |  from the Dockerfile. Stored in a        |
 *     |                  |  registry. Like an AMI for containers.   |
 *     +------------------+------------------------------------------+
 *     |  Container       |  A running instance of an Image.         |
 *     |                  |  Like an EC2 instance is a running       |
 *     |                  |  instance of an AMI.                     |
 *     +------------------+------------------------------------------+
 *
 *
 *   Docker workflow — from code to running container:
 *
 *    Step 1: Write Dockerfile (recipe)
 *    ┌─────────────────────────┐
 *    │  FROM node:18           │  ← start from Node.js 18 base image
 *    │  WORKDIR /app           │  ← set working directory
 *    │  COPY package*.json ./  │  ← copy dependency files
 *    │  RUN npm install        │  ← install dependencies
 *    │  COPY . .               │  ← copy all app code
 *    │  EXPOSE 3000            │  ← declare the port app uses
 *    │ CMD ["node","server.js"]│  ← command to start the app
 *    └─────────────────────────┘
 *             │
 *             │  docker build -t myapp:v1 .
 *             ▼
 *    Step 2: Image built and stored
 *    ┌─────────────────────────┐
 *    │  myapp:v1               │
 *    │  Packaged snapshot of   │
 *    │  app + all dependencies │
 *    │  Runs identically on    │
 *    │  any machine            │
 *    └─────────────────────────┘
 *             │
 *             │  docker run -p 3000:3000 myapp:v1
 *             ▼
 *    Step 3: Container running
 *    ┌─────────────────────────┐
 *    │  Container              │
 *    │  App running on port    │
 *    │  3000 ✓                 │
 *    │  Identical on any       │
 *    │  machine ✓              │
 *    └─────────────────────────┘
 *
 *
 *   Common Docker commands:
 *
 *     $ docker build -t myapp:v1 .          ← build image from Dockerfile
 *     $ docker run -p 3000:3000 myapp:v1    ← run container, map port 3000
 *     $ docker ps                           ← list all running containers
 *     $ docker stop <container-id>          ← stop a running container
 *     $ docker rm <container-id>            ← delete a stopped container
 *     $ docker images                       ← list all local images
 *     $ docker rmi myapp:v1                 ← delete an image
 *     $ docker logs <container-id>          ← view container logs
 *     $ docker exec -it <id> bash           ← open a shell inside container
 *
 *
 *   What is a Container Registry?
 *   - A registry is a storage place for Docker images.
 *   - Think of it like GitHub but for Docker images instead of code.
 *   - You push your image to a registry. Any server then pulls
 *     and runs that exact same image — guaranteed same environment.
 *
 *     +------------------+------------------------------------------+
 *     |  Registry        |  Description                             |
 *     +------------------+------------------------------------------+
 *     |  Docker Hub      |  Public registry. Free for public images.|
 *     |                  |  docker pull nginx fetches from here.    |
 *     +------------------+------------------------------------------+
 *     |  Amazon ECR      |  AWS's own private registry.             |
 *     |  (Elastic        |  Store your images privately in AWS.     |
 *     |  Container       |  Integrates natively with ECS.           |
 *     |  Registry)       |                                          |
 *     +------------------+------------------------------------------+
 *     |  GitHub GHCR     |  GitHub's own registry.                  |
 *     |                  |  Useful in CI/CD pipelines.              |
 *     +------------------+------------------------------------------+
 *
 *
 *   Push and pull flow:
 *
 *    Your Laptop                         Amazon ECR (Registry)
 *    ┌──────────────┐                    ┌───────────────────────┐
 *    │ docker build │                    │  myapp:v1  stored     │
 *    │ myapp:v1     │──── docker push ──►│  privately in AWS     │
 *    └──────────────┘                    └──────────┬────────────┘
 *                                                   │
 *                                         ECS pulls image
 *                                         ┌──────────┴────────────┐
 *                                         │                       │
 *                                    ┌────▼──────┐          ┌─────▼─────┐
 *                                    │ ECS Task 1│          │ ECS Task 2│
 *                                    │ myapp:v1  │          │ myapp:v1  │
 *                                    └───────────┘          └───────────┘
 *                                    Exact same image — identical everywhere ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * PART 4 — Why EC2 + Docker Alone is Still Not Enough
 * ───────────────────────────────────────────────────────
 *
 *   Containers solve the environment problem. But now we have
 *   a new set of questions when running at scale:
 *
 *   - You have 20 containers running across 5 EC2 instances.
 *     How do you know which container is on which machine?
 *   - A container crashes at 3 AM. Who restarts it?
 *   - Traffic spikes. How do you quickly add 5 more containers?
 *   - How do you deploy a new image version across all containers
 *     with zero downtime?
 *   - How do containers on different EC2 machines talk to each other?
 *
 *   Running containers manually on EC2 does not answer any of this.
 *   You need a tool that manages containers for you at scale.
 *   This is called a Container Orchestrator.
 *
 *
 *   The container management problem at scale:
 *
 *    ┌──────────────────────────────────────────────────────┐
 *    │  5 EC2 instances, each running containers            │
 *    │                                                      │
 *    │  EC2 #1: [Users] [Orders] [Payments]                 │
 *    │  EC2 #2: [Users] [Orders] [ML]                       │
 *    │  EC2 #3: [Users] [Payments] [ML]                     │
 *    │  EC2 #4: [Orders] [Payments]                         │
 *    │  EC2 #5: [Users] [ML]                                │
 *    │                                                      │
 *    │  Questions you can't answer manually:                │
 *    │  - Orders crashes on EC2 #2. Who restarts it? ✗      │
 *    │  - Need to deploy new Orders image to all. How? ✗    │
 *    │  - Traffic spikes — need 3 more Orders. Where? ✗     │
 *    │  - Which EC2 has spare capacity right now? ✗         │
 *    └──────────────────────────────────────────────────────┘
 *
 *    Managing this manually is impossible at scale.
 *    You need something to orchestrate it all. → ECS
 *
 *
 * ───────────────────────────────────────────────────────
 * ECS — Elastic Container Service
 * ───────────────────────────────────────────────────────
 *
 *   - ECS (Elastic Container Service) is AWS's fully managed
 *     container orchestration service.
 *   - It answers all the questions above — automatically.
 *   - You tell ECS: "I want 5 copies of my app container running
 *     at all times." ECS places them, monitors them, restarts
 *     them if they crash, and scales them based on traffic.
 *   - You stop managing individual EC2 instances and start thinking
 *     in terms of containers and services.
 *
 *
 *   EC2 way vs ECS way:
 *
 *    EC2 way:                          ECS way:
 *    "I manage 10 servers.             "I want 5 copies of my
 *     I SSH into each one.              container running at all times.
 *     I deploy manually.                ECS handles placement,
 *     I hope nothing breaks."           restarts, and scaling."
 *
 *
 *   Where ECS fits in the AWS stack:
 *
 *     +---------------------------------------------------------------+
 *     |                      [Your Application]                       |
 *     +---------------------------------------------------------------+
 *     |                Docker Container (runs your app)               |
 *     +---------------------------------------------------------------+
 *     |              ECS (manages and orchestrates containers)        |
 *     +---------------------------------------------------------------+
 *     |        EC2 Instances or Fargate (compute to run on)           |
 *     +---------------------------------------------------------------+
 *     |              Hypervisor (AWS Nitro System)                    |
 *     +---------------------------------------------------------------+
 *     |                      Physical Layer                           |
 *     +---------------------------------------------------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 1 — ECS Core Concepts
 * ───────────────────────────────────────────────────────
 *
 *   ECS has four key building blocks. You must understand all four
 *   before creating anything.
 *
 *
 *   Building Block 1 — Task Definition (the blueprint):
 *   - A Task Definition is a blueprint describing how to run
 *     your container — exactly like a Launch Template in EC2
 *     but for containers instead of servers.
 *   - You define: which Docker image to use, how much CPU and
 *     RAM to give it, which ports to expose, environment variables,
 *     and how to collect logs.
 *   - Task Definitions are versioned — every change creates a new
 *     revision. You can roll back to any previous version.
 *
 *     Task Definition contains:
 *     ┌───────────────────────────────────────────────────────────┐
 *     │  Image     : 123456.dkr.ecr.ap-south-1.amazonaws.com/     │
 *     │              myapp:v1   (pulled from ECR)                 │
 *     │  CPU       : 256  (0.25 vCPU)                             │
 *     │  Memory    : 512 MB                                       │
 *     │  Port      : 3000  (port your app listens on)             │
 *     │  Env Vars  : DB_HOST=rds-endpoint, NODE_ENV=production    │
 *     │  Logging   : awslogs → sends logs to CloudWatch           │
 *     └───────────────────────────────────────────────────────────┘
 *
 *
 *   Building Block 2 — Task (the running instance):
 *   - A Task is a running instance of a Task Definition.
 *   - Like an EC2 instance is a running instance of an AMI.
 *   - One Task = one or more containers running together.
 *   - Tasks are good for: batch jobs, one-off scripts, data processing
 *     — things that run, finish, and stop.
 *
 *
 *   Building Block 3 — Service (keeps Tasks alive):
 *   - A Service tells ECS to keep a specific number of Tasks
 *     running at all times.
 *   - If a Task crashes, the Service automatically starts a new one.
 *   - Services are for long-running apps: web servers, APIs,
 *     background workers — things that should never go down.
 *   - Services connect to Load Balancers and support Auto Scaling.
 *
 *     Service settings:
 *     ┌───────────────────────────────────────────────────────────┐
 *     │  Task Definition : myapp-task:3 (revision 3)              │
 *     │  Desired count   : 3  ← keep 3 Tasks running at all times │
 *     │  Min count       : 1                                      │
 *     │  Max count       : 10                                     │
 *     │  Load Balancer   : attached ALB Target Group              │
 *     └───────────────────────────────────────────────────────────┘
 *
 *
 *   Building Block 4 — Cluster (the container for everything):
 *   - A Cluster is the logical grouping of all your ECS resources.
 *   - Think of it as a folder that holds all your Services and Tasks.
 *   - One AWS account can have multiple clusters.
 *     e.g., chaicode-production and chaicode-staging.
 *
 *
 *   How all four fit together:
 *
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  ECS Cluster: chaicode-production                           │
 *    │                                                             │
 *    │  ┌───────────────────────┐  ┌───────────────────────┐       │
 *    │  │  Service: users-svc   │  │  Service: orders-svc  │       │
 *    │  │  Desired: 3 Tasks     │  │  Desired: 5 Tasks     │       │
 *    │  │                       │  │                       │       │
 *    │  │  ┌──────┐ ┌──────┐    │  │  ┌──────┐ ┌──────┐    │       │
 *    │  │  │Task 1│ │Task 2│    │  │  │Task 1│ │Task 2│    │       │
 *    │  │  │      │ │Task 3│    │  │  │Task 3│ │Task 4│    │       │
 *    │  │  └──────┘ └──────┘    │  │  │Task 5│ └──────┘    │       │
 *    │  │                       │  │  └──────┘             │       │
 *    │  └───────────────────────┘  └───────────────────────┘       │
 *    └─────────────────────────────────────────────────────────────┘
 *
 *    Task crashes → Service detects it → starts a replacement  ✓
 *    Traffic spikes → Service scales up Task count              ✓
 *
 *
 *   ECS concepts compared to EC2 concepts:
 *
 *     +---------------------+--------------------+--------------------+
 *     |  Concept            |  EC2 equivalent    |  ECS equivalent    |
 *     +---------------------+--------------------+--------------------+
 *     |  Blueprint          |  Launch Template   |  Task Definition   |
 *     |  Running instance   |  EC2 Instance      |  Task              |
 *     |  Keep N running     |  Auto Scaling Group|  ECS Service       |
 *     |  Logical grouping   |  (none)            |  ECS Cluster       |
 *     |  Image to run from  |  AMI               |  Docker Image      |
 *     |  Image storage      |  (none / S3)       |  ECR               |
 *     +---------------------+--------------------+--------------------+
 *
 *
 * ──────────────────────────────────────────────────────────────
 * STEP 2 — ECS Launch Types (Where Do Containers Actually Run?)
 * ──────────────────────────────────────────────────────────────
 *
 *   ECS needs actual compute (servers) to run your containers on.
 *   AWS gives you two choices:
 *
 *     +------------------+--------------------+--------------------+
 *     |  Feature         |  EC2 Launch Type   |  Fargate           |
 *     +------------------+--------------------+--------------------+
 *     |  You manage      |  EC2 instances     |  Nothing ✓         |
 *     |  AWS manages     |  Container placing |  Everything        |
 *     |  Control         |  Full control      |  No server access  |
 *     |  Cost            |  Pay per instance  |  Pay per container |
 *     |  Best for        |  GPU, custom needs |  Most workloads    |
 *     +------------------+--------------------+--------------------+
 *
 *
 *   Launch Type 1 — EC2 Launch Type:
 *   - You provision and manage EC2 instances yourself.
 *   - ECS places containers on those instances for you.
 *   - You are responsible for: instance types, patching, capacity.
 *   - More control, more work.
 *
 *    ┌─────────────────────────────────────────────────────┐
 *    │  ECS Cluster (EC2 Launch Type)                      │
 *    │                                                     │
 *    │  ┌──────────────────┐    ┌──────────────────┐       │
 *    │  │   EC2 Instance 1 │    │   EC2 Instance 2 │       │
 *    │  │  ┌─────┐ ┌─────┐ │    │  ┌─────┐ ┌─────┐ │       │
 *    │  │  │ Con │ │ Con │ │    │  │ Con │ │ Con │ │       │
 *    │  │  │  1  │ │  2  │ │    │  │  3  │ │  4  │ │       │
 *    │  │  └─────┘ └─────┘ │    │  └─────┘ └─────┘ │       │
 *    │  │  you manage this │    │  you manage this │       │
 *    │  └──────────────────┘    └──────────────────┘       │
 *    │           ↑                      ↑                  │
 *    │     ECS places containers on your EC2 instances     │
 *    └─────────────────────────────────────────────────────┘
 *
 *
 *   Launch Type 2 — Fargate (Serverless Containers):
 *   - Fargate is the easiest way to run containers on AWS.
 *   - You do not manage any EC2 instances at all.
 *   - You just define your Task (CPU, RAM, image) and AWS
 *     provisions the compute, runs the container, and cleans up.
 *   - Pay only for the exact CPU and RAM your container uses.
 *   - No patching, no capacity planning, no SSH into servers.
 *   - Recommended for most teams starting with ECS.
 *
 *    ┌─────────────────────────────────────────────────────┐
 *    │  ECS Cluster (Fargate)                              │
 *    │                                                     │
 *    │  ┌───────────┐  ┌───────────┐  ┌───────────┐        │
 *    │  │ Container │  │ Container │  │ Container │        │
 *    │  │     1     │  │     2     │  │     3     │        │
 *    │  └───────────┘  └───────────┘  └───────────┘        │
 *    │        ↑               ↑               ↑            │
 *    │        AWS manages the servers underneath           │
 *    │        You never see or touch them                  │
 *    └─────────────────────────────────────────────────────┘
 *
 *
 *   Which launch type should I use?
 *
 *    Do you need custom EC2 types, GPU,
 *    or specific kernel settings?
 *         │
 *    ┌────┴────┐
 *    │         │
 *   YES        NO
 *    │         │
 *    ▼         ▼
 *   EC2      Fargate — no servers
 *   Launch   to manage ✓
 *   Type
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 3 — Amazon ECR (Elastic Container Registry)
 * ───────────────────────────────────────────────────────
 *
 *   - ECR is AWS's private Docker image registry.
 *   - Before ECS can run your container, it needs to pull the
 *     Docker image from somewhere. That somewhere is ECR.
 *   - ECR is private — only your AWS account can access it
 *     unless you explicitly make it public.
 *   - ECR integrates natively with ECS — no extra auth setup needed.
 *
 *
 *   ECR workflow:
 *
 *    Your Laptop / CI Pipeline
 *    ┌──────────────────────────────────────────────────────┐
 *    │  1. docker build -t myapp:v1 .                       │
 *    │  2. docker tag myapp:v1 <ecr-url>/myapp:v1           │
 *    │  3. docker push <ecr-url>/myapp:v1                   │
 *    └──────────────────────────────┬───────────────────────┘
 *                                   │ image pushed to ECR
 *                                   │
 *    Amazon ECR                     ▼
 *    ┌──────────────────────────────────────────────────────┐
 *    │  Repository: myapp                                   │
 *    │  Tags stored:                                        │
 *    │  myapp:v1  ← latest push                             │
 *    │  myapp:v2  ← previous version                        │
 *    │  myapp:v3  ← older version                           │
 *    └──────────────────────────────┬───────────────────────┘
 *                                   │ ECS pulls image automatically
 *                                   ▼
 *    ECS Task starts running container ✓
 *
 *
 *   How to create an ECR repository and push an image:
 *
 *     AWS Console
 *       → Search "ECR" → "Elastic Container Registry"
 *       → Click "Create Repository"
 *       → Visibility  : Private
 *       → Name        : myapp
 *       → Click "Create Repository"
 *       → Click "View push commands" → AWS gives you exact commands
 *
 *     Push commands (shown by AWS):
 *     $ aws ecr get-login-password --region ap-south-1 | \
 *       docker login --username AWS --password-stdin \
 *       <account-id>.dkr.ecr.ap-south-1.amazonaws.com
 *
 *     $ docker build -t myapp .
 *     $ docker tag myapp:latest \
 *       <account-id>.dkr.ecr.ap-south-1.amazonaws.com/myapp:latest
 *     $ docker push \
 *       <account-id>.dkr.ecr.ap-south-1.amazonaws.com/myapp:latest
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 4 — Create and Deploy on ECS (step by step)
 * ───────────────────────────────────────────────────────
 *
 *   PART A — Create an ECS Cluster:
 *
 *     AWS Console
 *       → Search "ECS" → "Elastic Container Service"
 *       → Click "Clusters" → "Create Cluster"
 *       → Cluster name   : chaicode-production
 *       → Infrastructure : AWS Fargate (recommended)
 *       → Click "Create"
 *
 *
 *   PART B — Create a Task Definition:
 *   (This is the blueprint — what image, how much CPU, which ports)
 *
 *     ECS Console
 *       → Click "Task Definitions" → "Create new Task Definition"
 *       → Task Definition name : myapp-task
 *       → Launch type          : AWS Fargate
 *       → CPU                  : 0.5 vCPU
 *       → Memory               : 1 GB
 *       → Add Container:
 *           Container name : myapp
 *           Image URI      : <ecr-url>/myapp:latest
 *           Port mappings  : 3000  (container port)
 *           Environment variables:
 *             NODE_ENV = production
 *             DB_HOST  = <rds-endpoint>
 *           Log configuration:
 *             Driver : awslogs
 *             Group  : /ecs/myapp
 *       → Click "Create"
 *
 *
 *   PART C — Create an ECS Service:
 *   (This keeps your Tasks running and connects to the Load Balancer)
 *
 *     ECS Console
 *       → Select your Cluster (chaicode-production)
 *       → Click "Services" → "Create"
 *       → Launch type        : Fargate
 *       → Task Definition    : myapp-task (latest revision)
 *       → Service name       : myapp-service
 *       → Desired Tasks      : 3
 *       → VPC                : select your VPC
 *       → Subnets            : select at least 2 (different AZs)
 *       → Security Group     : allow inbound on port 3000
 *       → Load Balancer      : attach existing ALB + Target Group
 *       → Auto Scaling       :
 *           Min tasks   : 1
 *           Max tasks   : 10
 *           Metric      : ECSServiceAverageCPUUtilization
 *           Target      : 50%
 *       → Click "Create Service"
 *
 *
 *   PART D — Verify deployment:
 *
 *     ECS Console → Clusters → chaicode-production
 *       → Services → myapp-service
 *       → "Tasks" tab   → all tasks should show: RUNNING ✓
 *       → "Events" tab  → shows deployment progress and any errors
 *       → "Logs" tab    → container logs via CloudWatch
 *
 *     ALB check:
 *     → EC2 Console → Target Groups → select yours
 *     → All targets should show: healthy ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 5 — Deploying a New Version (Zero Downtime)
 * ───────────────────────────────────────────────────────
 *
 *   This is where ECS shines over EC2. Deploying a new version
 *   requires zero manual SSH, zero downtime, and is fully automated.
 *
 *
 *   Steps to deploy v2 of your app:
 *
 *   1. Build and push the new image to ECR:
 *      $ docker build -t myapp:v2 .
 *      $ docker push <ecr-url>/myapp:v2
 *
 *   2. Create a new Task Definition revision pointing to v2:
 *      ECS Console → Task Definitions → myapp-task
 *        → "Create new revision"
 *        → Update image URI to myapp:v2
 *        → Click "Create"  (creates revision 2)
 *
 *   3. Update the Service to use the new revision:
 *      ECS Console → Services → myapp-service
 *        → Click "Update"
 *        → Task Definition: select latest revision
 *        → Deployment type : Rolling update
 *        → Click "Update Service"
 *
 *
 *   How Rolling Update works (zero downtime):
 *
 *    Before — 3 Tasks running v1:
 *    [Task:v1] [Task:v1] [Task:v1]   ← all serving traffic
 *
 *    ECS starts a new v2 Task:
 *    [Task:v1] [Task:v1] [Task:v1] [Task:v2 starting...]
 *
 *    v2 Task passes health check → ALB adds it to rotation:
 *    [Task:v1] [Task:v1] [Task:v1] [Task:v2 ✓]
 *
 *    ECS drains and stops one v1 Task:
 *    [Task:v1] [Task:v1] [Task:v2 ✓]
 *
 *    ECS repeats until all Tasks are v2:
 *    [Task:v2] [Task:v2] [Task:v2]  ✓ deployment complete
 *
 *    Users never hit an unavailable container — zero downtime ✓
 *    If v2 fails health check → ECS stops the rollout and alerts ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 6 — ECS Auto Scaling
 * ───────────────────────────────────────────────────────
 *
 *   - ECS Services support Auto Scaling just like EC2 ASG.
 *   - Instead of scaling EC2 instances, ECS scales Tasks (containers).
 *   - Traffic goes up → ECS launches more Tasks automatically.
 *   - Traffic drops → ECS terminates extra Tasks automatically.
 *   - With Fargate, no EC2 instances to manage at all — AWS
 *     provisions the underlying compute automatically per Task.
 *
 *
 *   ECS Auto Scaling in action:
 *
 *    Traffic normal — CPU at 40%:
 *    [Task] [Task] [Task]  ← 3 Tasks running
 *
 *    Traffic spikes — CPU hits 70%:
 *    Auto Scaling triggers → launch 2 more Tasks
 *    [Task] [Task] [Task] [Task] [Task]  ← 5 Tasks, CPU drops to 42%
 *
 *    Traffic drops — CPU drops to 15%:
 *    Auto Scaling triggers → remove 2 Tasks
 *    [Task] [Task] [Task]  ← back to 3 Tasks, cost saved ✓
 *
 *
 *   ECS vs EC2 Auto Scaling:
 *
 *     +------------------+--------------------+--------------------+
 *     |  Feature         |  EC2 Auto Scaling  |  ECS Auto Scaling  |
 *     +------------------+--------------------+--------------------+
 *     |  What scales     |  EC2 instances     |  Containers/Tasks  |
 *     |  Speed           |  Minutes (boot OS) |  Seconds (pull img)|
 *     |  Granularity     |  Whole server      |  Individual service |
 *     |  Cost unit       |  Per instance/hour |  Per vCPU + GB/sec |
 *     +------------------+--------------------+--------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 7 — ECS with Load Balancer
 * ───────────────────────────────────────────────────────
 *
 *   - ECS integrates directly with ALB.
 *   - When a Task starts, ECS automatically registers it as a
 *     target in the ALB Target Group.
 *   - When a Task stops, ECS automatically deregisters it.
 *   - No manual steps — it is all automatic.
 *
 *
 *   ECS + ALB traffic flow:
 *
 *    [User] → chaicode.com
 *         │
 *         ▼
 *    [GoDaddy DNS]
 *    CNAME → ALB DNS name
 *         │
 *         ▼
 *    [ALB — Application Load Balancer]
 *    Listens on port 80/443
 *         │
 *         ┌──────────┬──────────┐
 *         │          │          │
 *         ▼          ▼          ▼
 *    [ECS Task 1] [ECS Task 2] [ECS Task 3]
 *    Container    Container    Container
 *    port 3000    port 3000    port 3000
 *         │          │          │
 *         └──────────┴──────────┘
 *                    │
 *                    ▼
 *              [Database — RDS]
 *
 *
 *   ECS with path-based routing (microservices):
 *
 *    [ALB]
 *      │
 *      ├── /api/users    → ECS Service: users-svc    (3 Tasks)
 *      ├── /api/orders   → ECS Service: orders-svc   (5 Tasks)
 *      └── /api/payments → ECS Service: payments-svc (2 Tasks)
 *
 *    Each service scales independently based on its own traffic ✓
 *    Orders service busy? Scale only orders-svc. Others untouched. ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 8 — ECS Task Lifecycle
 * ───────────────────────────────────────────────────────
 *
 *    [Service decides to start a new Task]
 *           │
 *           ▼
 *    [PROVISIONING]  ← ECS allocating resources (Fargate only)
 *           │
 *           ▼
 *    [PENDING]  ← container image being pulled from ECR
 *           │
 *           ▼
 *    [RUNNING]  ← container is live, billing starts (Fargate)
 *           │
 *      ┌────┴──────────────────┐
 *      │                       │
 *      ▼                       ▼
 *  [STOPPED]            [DEPROVISIONING]
 *  App crashed          ECS cleaning up
 *  or job finished      resources
 *      │                       │
 *      ▼                       ▼
 *  Service detects        [DELETED]
 *  stopped Task →
 *  starts replacement
 *  automatically ✓
 *
 *
 *   Billing (Fargate):
 *   - RUNNING  → billed per vCPU-second and GB-second used
 *   - STOPPED  → no billing
 *   - Much more granular than EC2 — pay for exactly what you use
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 9 — Delete ECS Resources (Clean Up)
 * ───────────────────────────────────────────────────────
 *
 *   Always clean up in this order to avoid charges:
 *
 *   PART A — Scale Service down to 0 Tasks first:
 *
 *     ECS Console → Cluster → Service → "Update"
 *       → Desired Tasks : 0
 *       → Click "Update"
 *       → Wait for all Tasks to reach status: STOPPED
 *
 *   PART B — Delete the Service:
 *
 *     ECS Console → Cluster → Services
 *       → Select service → "Delete Service" → Confirm
 *
 *   PART C — Deregister the Task Definition:
 *   (Task Definitions cannot be deleted, only deregistered)
 *
 *     ECS Console → Task Definitions → select all revisions
 *       → Click "Deregister" → Confirm
 *
 *   PART D — Delete the Cluster:
 *
 *     ECS Console → Clusters → select cluster
 *       → Click "Delete Cluster" → Confirm
 *
 *   PART E — Delete ECR Images and Repository:
 *
 *     ECR Console → Repositories → select repository
 *       → Select images → "Delete"
 *       → To delete whole repository → "Delete Repository"
 *
 *   Clean up order (summary):
 *   1. Scale Service → 0 Tasks
 *   2. Delete Service
 *   3. Deregister Task Definition
 *   4. Delete Cluster
 *   5. Delete ECR images / repository
 *   6. Delete ALB Target Group (if created separately for ECS)
 *
 *
 * ───────────────────────────────────────────────────────
 * FINAL ARCHITECTURE — ECS + Fargate + ALB + ECR
 * ───────────────────────────────────────────────────────
 *
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │                    INTERNET                                  │
 *    └────────────────────────┬────────────────────────────────────┘
 *                             │  User types chaicode.com
 *                             ▼
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  GoDaddy DNS                                                 │
 *    │  CNAME: chaicode.com → ALB DNS name                         │
 *    └────────────────────────┬────────────────────────────────────┘
 *                             │
 *                             ▼
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  ALB (Application Load Balancer)                             │
 *    │  SSL termination | Path routing | Health checks              │
 *    └──────┬────────────────┬──────────────────┬──────────────────┘
 *           │                │                  │
 *      /api/users       /api/orders        /api/payments
 *           │                │                  │
 *           ▼                ▼                  ▼
 *    ┌─────────────┐  ┌─────────────┐  ┌─────────────┐
 *    │ ECS Service │  │ ECS Service │  │ ECS Service │
 *    │ users-svc   │  │ orders-svc  │  │ payments-svc│
 *    │             │  │             │  │             │
 *    │ [T][T][T]   │  │[T][T][T]   │  │ [T][T]      │
 *    │ 3 Tasks     │  │[T][T]       │  │ 2 Tasks     │
 *    │             │  │ 5 Tasks     │  │             │
 *    └──────┬──────┘  └──────┬──────┘  └──────┬──────┘
 *           │                │                 │
 *           └────────────────┴─────────────────┘
 *                            │
 *                            ▼
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  Database Layer (RDS / DynamoDB)                             │
 *    │  Only accessible from ECS Security Group — not internet      │
 *    └─────────────────────────────────────────────────────────────┘
 *
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  Supporting Services                                         │
 *    │                                                             │
 *    │  Amazon ECR   → stores all Docker images privately          │
 *    │  CloudWatch   → collects all container logs                 │
 *    │  Fargate      → serverless compute (no EC2 to manage)       │
 *    │  IAM Role     → gives ECS permission to pull from ECR       │
 *    └─────────────────────────────────────────────────────────────┘
 *
 *
 *   EC2 vs ECS — when to use which:
 *
 *     +---------------------------+----------------+----------------+
 *     |  Situation                |  Use EC2       |  Use ECS       |
 *     +---------------------------+----------------+----------------+
 *     |  Simple single app        |  ✓             |                |
 *     |  Need full server control |  ✓             |                |
 *     |  GPU workloads            |  ✓             |                |
 *     |  Multiple services        |                |  ✓             |
 *     |  Consistent environments  |                |  ✓             |
 *     |  Fast deploys (no SSH)    |                |  ✓             |
 *     |  Scale one service only   |                |  ✓             |
 *     |  Avoid managing servers   |                |  ✓ Fargate     |
 *     +---------------------------+----------------+----------------+
 *
 *
 *   Summary — what each component does:
 *
 *     +------------------+------------------------------------------+
 *     |  Component       |  Responsibility                           |
 *     +------------------+------------------------------------------+
 *     |  Docker          |  Packages app + dependencies into image  |
 *     +------------------+------------------------------------------+
 *     |  Dockerfile      |  Recipe to build the Docker image        |
 *     +------------------+------------------------------------------+
 *     |  ECR             |  Stores Docker images privately in AWS   |
 *     +------------------+------------------------------------------+
 *     |  Task Definition |  Blueprint: image, CPU, RAM, ports, env  |
 *     +------------------+------------------------------------------+
 *     |  Task            |  One running container instance          |
 *     +------------------+------------------------------------------+
 *     |  Service         |  Keeps N Tasks alive, restarts crashes   |
 *     +------------------+------------------------------------------+
 *     |  Cluster         |  Logical grouping of all ECS resources   |
 *     +------------------+------------------------------------------+
 *     |  Fargate         |  Serverless compute — no EC2 to manage  |
 *     +------------------+------------------------------------------+
 *     |  ALB             |  Routes user traffic to healthy Tasks    |
 *     +------------------+------------------------------------------+
 *     |  Auto Scaling    |  Adds/removes Tasks based on traffic     |
 *     +------------------+------------------------------------------+
 */

In [ ]:
/**
 * ───────────────────────────────────────────────────────
 * From ECS to Lambda — Why We Need Serverless
 * ───────────────────────────────────────────────────────
 *
 * ───────────────────────────────────────────────────────
 * PART 1 — The Problems with Always-On Servers
 * ───────────────────────────────────────────────────────
 *
 *   We have learned EC2 (virtual machines) and ECS (containers).
 *   Both are great — but they share one fundamental problem:
 *   you pay for them even when nobody is using your app.
 *
 *
 *   Problem 1 — You pay 24/7 even for idle time
 *
 *   - Your EC2 or ECS container runs 24 hours a day.
 *   - But your app might only receive requests for 2 hours a day.
 *   - The other 22 hours you are paying for a server doing nothing.
 *
 *    Usage pattern of a typical internal tool:
 *
 *    Traffic
 *    │
 *    │         ███████████
 *    │         █         █
 *    │         █         █
 *    │_________█_________█_________________________
 *    └─────────────────────────────────────────────► time
 *    12am  3am  9am       5pm                  12am
 *             working hours only
 *
 *    EC2 / ECS billing:
 *    ████████████████████████████████████████████  ← paying all 24 hrs
 *
 *    Lambda billing:
 *    _________████████████_________________________  ← paying only when used
 *
 *
 *   Problem 2 — You manage infrastructure you don't need to
 *
 *   - With EC2 you manage servers.
 *   - With ECS you manage containers.
 *   - For many use cases (send an email, resize an image,
 *     process a webhook) you don't want to manage any of this.
 *   - You just want to write a function and run it.
 *
 *
 *   Problem 3 — Over-provisioning for spikes
 *
 *   - You keep 3 EC2 instances running to handle traffic spikes.
 *   - 99% of the time only 1 is needed.
 *   - The other 2 sit idle, costing money.
 *   - Auto Scaling helps but it takes minutes to add new instances.
 *     Lambda scales in milliseconds.
 *
 *
 *   Summary — when EC2 / ECS is overkill:
 *
 *     +-----------------------------+----------------------------------+
 *     |  Task                       |  Do you really need a server?    |
 *     +-----------------------------+----------------------------------+
 *     |  Send a welcome email       |  No — just run a function        |
 *     |  Resize an uploaded image   |  No — just run a function        |
 *     |  Process a webhook          |  No — just run a function        |
 *     |  Run a nightly report       |  No — just run a function        |
 *     |  Validate a form submission |  No — just run a function        |
 *     |  A full web application     |  Yes — use EC2 / ECS             |
 *     |  A database server          |  Yes — use EC2 / RDS             |
 *     +-----------------------------+----------------------------------+
 *
 *
 * ───────────────────────────────────────────────────────
 * AWS Lambda — Serverless Functions
 * ───────────────────────────────────────────────────────
 *
 *   - AWS Lambda lets you run code without provisioning or managing
 *     any servers at all.
 *   - You upload your code (a function). AWS runs it when triggered.
 *   - You pay only for the milliseconds your code actually runs.
 *   - When nobody is calling your function, you pay nothing.
 *   - Lambda automatically scales — from 1 request to 10,000
 *     concurrent requests — with no configuration needed.
 *
 *
 *   The core idea — event driven execution:
 *
 *    Something happens (an event)
 *         │
 *         ▼
 *    Lambda function runs
 *         │
 *         ▼
 *    Function finishes
 *         │
 *         ▼
 *    Lambda shuts down — you stop being billed
 *
 *
 *   EC2 / ECS vs Lambda:
 *
 *     +------------------+--------------------+--------------------+
 *     |  Feature         |  EC2 / ECS         |  Lambda            |
 *     +------------------+--------------------+--------------------+
 *     |  Manage servers  |  Yes               |  No                |
 *     |  Always running  |  Yes               |  No — on demand    |
 *     |  Billing         |  Per hour/second   |  Per millisecond   |
 *     |  Idle cost       |  Yes               |  Zero              |
 *     |  Scale speed     |  Minutes           |  Milliseconds      |
 *     |  Max run time    |  Unlimited         |  15 minutes        |
 *     |  Best for        |  Long running apps |  Short tasks       |
 *     +------------------+--------------------+--------------------+
 *
 *
 *   Where Lambda fits in the AWS stack:
 *
 *     ┌───────────────────────────────────────────────────────┐
 *     │              [Your Function Code]                     │
 *     ├───────────────────────────────────────────────────────┤
 *     │         Lambda Runtime (Node.js, Python, etc.)        │
 *     ├───────────────────────────────────────────────────────┤
 *     │           AWS Managed Infrastructure                  │
 *     │     (you never see or touch any of this)              │
 *     ├───────────────────────────────────────────────────────┤
 *     │                  Physical Layer                       │
 *     └───────────────────────────────────────────────────────┘
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 1 — Lambda Core Concepts
 * ───────────────────────────────────────────────────────
 *
 *   Lambda has a few key terms you need to understand first.
 *
 *
 *   Concept 1 — Function:
 *   - A Function is your actual code — the logic you want to run.
 *   - It can be written in: Node.js, Python, Java, Go, Ruby,
 *     .NET, or a custom runtime.
 *   - Every Lambda function has a Handler — the entry point
 *     that AWS calls when the function is triggered.
 *
 *     Simple Node.js Lambda function:
 *
 *     exports.handler = async (event) => {
 *       console.log('Event received:', event);
 *       return {
 *         statusCode: 200,
 *         body: JSON.stringify({ message: 'Hello from Lambda!' })
 *       };
 *     };
 *
 *     - event   → the data passed in when the function is triggered
 *     - handler → the function AWS calls (exports.handler)
 *     - return  → the response Lambda sends back
 *
 *
 *   Concept 2 — Trigger (what starts the function):
 *   - A Trigger is the event that causes Lambda to run your function.
 *   - Lambda can be triggered by many AWS services:
 *
 *     +---------------------+---------------------------------------+
 *     |  Trigger            |  When it fires                        |
 *     +---------------------+---------------------------------------+
 *     |  API Gateway        |  HTTP request hits your API           |
 *     |  S3                 |  File uploaded to a bucket            |
 *     |  DynamoDB Streams   |  Record inserted/updated/deleted      |
 *     |  SQS                |  Message added to a queue             |
 *     |  SNS                |  Notification published               |
 *     |  CloudWatch Events  |  Scheduled time (cron job)            |
 *     |  ALB                |  HTTP request from load balancer      |
 *     |  Cognito            |  User signs up or signs in            |
 *     |  EventBridge        |  Custom or AWS service event          |
 *     +---------------------+---------------------------------------+
 *
 *
 *   Concept 3 — Execution Environment:
 *   - When Lambda runs your function, AWS spins up a tiny isolated
 *     environment (a micro VM) just for your function.
 *   - This environment contains your code, the runtime,
 *     and any dependencies you packaged.
 *   - After your function finishes, AWS may keep the environment
 *     warm for a short time to handle the next request faster.
 *     This is called a Warm Start.
 *   - If no request comes, the environment is shut down.
 *     The next request starts a fresh environment — Cold Start.
 *
 *
 *   Cold Start vs Warm Start:
 *
 *    Cold Start (first request or after idle):
 *    [Request arrives]
 *         │
 *         ▼
 *    [AWS spins up environment]  ← takes ~100ms to 1 second
 *         │
 *         ▼
 *    [Download and initialise your code]
 *         │
 *         ▼
 *    [Run your function]
 *         │
 *         ▼
 *    [Return response]  ← total: slower
 *
 *    Warm Start (subsequent requests):
 *    [Request arrives]
 *         │
 *         ▼
 *    [Environment already running — skip setup]
 *         │
 *         ▼
 *    [Run your function directly]
 *         │
 *         ▼
 *    [Return response]  ← total: very fast
 *
 *    Cold Start impact:
 *    ┌──────────────────────────────────────────────┐
 *    │  Language     │  Typical Cold Start Time     │
 *    ├──────────────────────────────────────────────┤
 *    │  Node.js      │  ~100–200ms                  │
 *    │  Python       │  ~100–200ms                  │
 *    │  Java         │  ~500ms–2 seconds            │
 *    │  .NET         │  ~500ms–1 second             │
 *    │  Go           │  ~50–100ms                   │
 *    └──────────────────────────────────────────────┘
 *    For most APIs, cold starts are acceptable.
 *    For real-time or latency-sensitive apps, use Provisioned
 *    Concurrency to keep environments pre-warmed.
 *
 *
 *   Concept 4 — Concurrency:
 *   - Concurrency = how many instances of your function run
 *     at the same time.
 *   - If 100 requests arrive simultaneously, Lambda runs
 *     100 instances of your function in parallel automatically.
 *   - Default limit: 1,000 concurrent executions per region.
 *     (Can be increased by requesting a limit increase from AWS.)
 *   - Reserved Concurrency: cap a function to a max number to
 *     prevent it from consuming all available concurrency.
 *
 *    Concurrency scaling visual:
 *
 *    1 request:    [Function]
 *
 *    10 requests:  [Fn][Fn][Fn][Fn][Fn][Fn][Fn][Fn][Fn][Fn]
 *
 *    1000 requests:[Fn x 1000 running simultaneously]
 *                   AWS handles all of this automatically ✓
 *
 *
 *   Concept 5 — Lambda Layers:
 *   - A Layer is a zip file containing shared libraries or code
 *     that multiple Lambda functions can use.
 *   - Instead of packaging the same library in every function,
 *     you put it in a Layer and attach it.
 *   - e.g., a shared database utility, common validation code,
 *     or a large library like pandas for Python.
 *
 *    Without Layers:                With Layers:
 *    ┌──────────┐                   ┌──────────┐  ┌──────────┐
 *    │Function A│                   │Function A│  │Function B│
 *    │+ pandas  │                   └────┬─────┘  └────┬─────┘
 *    │(50 MB)   │                        │              │
 *    └──────────┘                        └──────┬───────┘
 *    ┌──────────┐                               │
 *    │Function B│                        ┌──────▼──────┐
 *    │+ pandas  │                        │   Layer     │
 *    │(50 MB)   │                        │   pandas    │
 *    └──────────┘                        │   (50 MB)   │
 *                                        └─────────────┘
 *    Each function = 50 MB              Shared once ✓
 *    Total = 100 MB                     Total = 50 MB + 2 small fns
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 2 — Lambda Supported Runtimes
 * ───────────────────────────────────────────────────────
 *
 *   Lambda supports these languages out of the box:
 *
 *     +------------------+------------------------------------------+
 *     |  Language        |  Runtime identifier                      |
 *     +------------------+------------------------------------------+
 *     |  Node.js 20      |  nodejs20.x                              |
 *     |  Node.js 18      |  nodejs18.x                              |
 *     |  Python 3.12     |  python3.12                              |
 *     |  Python 3.11     |  python3.11                              |
 *     |  Java 21         |  java21                                  |
 *     |  Java 17         |  java17                                  |
 *     |  .NET 8          |  dotnet8                                 |
 *     |  Go 1.x          |  provided.al2023 (custom runtime)        |
 *     |  Ruby 3.3        |  ruby3.3                                 |
 *     |  Custom Runtime  |  provided.al2023 (any language)          |
 *     +------------------+------------------------------------------+
 *
 *   For most web developers: Node.js or Python are the best choice.
 *   They have the fastest cold starts and largest community support.
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 3 — Lambda Pricing
 * ───────────────────────────────────────────────────────
 *
 *   Lambda pricing has two components:
 *
 *   a. Number of requests:
 *      - First 1 million requests per month: FREE
 *      - After that: $0.20 per 1 million requests
 *
 *   b. Duration (compute time):
 *      - Billed per millisecond of execution time
 *      - Based on the memory you allocate
 *      - First 400,000 GB-seconds per month: FREE
 *      - After that: $0.0000166667 per GB-second
 *
 *
 *   Pricing example:
 *
 *    Function runs 1 million times per month
 *    Each run takes 200ms and uses 128 MB RAM
 *
 *    Requests cost:
 *    1M requests = FREE (within free tier)
 *
 *    Duration cost:
 *    Duration = 1,000,000 × 0.2 seconds × 0.128 GB
 *             = 25,600 GB-seconds
 *    25,600 GB-seconds = FREE (within free tier)
 *
 *    Total cost = $0.00 per month ✓
 *
 *    This is why Lambda is extremely cost-effective for
 *    low-to-medium traffic workloads.
 *
 *
 *   Lambda vs EC2 cost comparison:
 *
 *     +------------------+--------------------+--------------------+
 *     |  Scenario        |  EC2 (t3.micro)    |  Lambda            |
 *     +------------------+--------------------+--------------------+
 *     |  Always running  |  ~$8/month         |  N/A               |
 *     |  10K req/month   |  ~$8/month         |  ~$0.00/month      |
 *     |  1M req/month    |  ~$8/month         |  ~$0.00/month      |
 *     |  100M req/month  |  ~$8-16/month      |  ~$2-5/month       |
 *     |  1B req/month    |  ~$50+/month       |  ~$200/month       |
 *     +------------------+--------------------+--------------------+
 *     Lambda wins at low-medium traffic. EC2 wins at very high
 *     sustained traffic where the server is always busy.
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 4 — Lambda Limits
 * ───────────────────────────────────────────────────────
 *
 *   Lambda has hard limits you must design around:
 *
 *     +---------------------------+----------------------------------+
 *     |  Limit                    |  Value                           |
 *     +---------------------------+----------------------------------+
 *     |  Max execution time       |  15 minutes                      |
 *     |  Memory                   |  128 MB to 10,240 MB (10 GB)     |
 *     |  CPU                      |  Scales with memory automatically|
 *     |                           |  10 GB RAM = ~6 vCPU             |
 *     |  Deployment package size  |  50 MB (zip) / 250 MB unzipped  |
 *     |  /tmp storage             |  512 MB to 10 GB                 |
 *     |  Concurrent executions    |  1,000 per region (default)      |
 *     |  Environment variables    |  4 KB total                      |
 *     |  Payload (request/resp)   |  6 MB synchronous                |
 *     |                           |  256 KB asynchronous             |
 *     +---------------------------+----------------------------------+
 *
 *   What Lambda is NOT good for:
 *   - Tasks that run longer than 15 minutes → use EC2 / ECS / Fargate
 *   - Apps that need persistent connections (WebSocket servers)
 *     → use EC2 / ECS (though API Gateway supports WebSockets)
 *   - Heavy compute that needs consistent performance
 *     → use EC2 with dedicated instance types
 *   - Apps that need a file system → use EC2 with EBS
 *     (Lambda /tmp is temporary and small)
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 5 — Create a Lambda Function (step by step)
 * ───────────────────────────────────────────────────────
 *
 *   PART A — Create the Function:
 *
 *     AWS Console
 *       → Search "Lambda"
 *       → Click "Create Function"
 *       → Choose: "Author from scratch"
 *       → Function name  : hello-world
 *       → Runtime         : Node.js 20.x
 *       → Architecture    : x86_64
 *       → Execution role  : Create a new role with basic Lambda permissions
 *           (this gives Lambda permission to write logs to CloudWatch)
 *       → Click "Create Function"
 *
 *
 *   PART B — Write your Function code:
 *
 *     AWS Console → Lambda → your function → "Code" tab
 *
 *     index.mjs (handler file):
 *
 *     export const handler = async (event) => {
 *       console.log('Received event:', JSON.stringify(event, null, 2));
 *
 *       const name = event.queryStringParameters?.name || 'World';
 *
 *       return {
 *         statusCode: 200,
 *         headers: { 'Content-Type': 'application/json' },
 *         body: JSON.stringify({ message: `Hello, ${name}!` })
 *       };
 *     };
 *
 *     → Click "Deploy" to save your code
 *
 *
 *   PART C — Configure Function settings:
 *
 *     AWS Console → Lambda → your function → "Configuration" tab
 *
 *     General Configuration:
 *       → Memory     : 128 MB  (increase for CPU-heavy tasks)
 *       → Timeout    : 3 seconds  (increase for slow tasks, max 15 min)
 *       → Description: "My first Lambda function"
 *
 *     Environment Variables:
 *       → DB_HOST    = rds-endpoint
 *       → NODE_ENV   = production
 *       → API_KEY    = your-secret-key
 *       (stored encrypted — safe for secrets)
 *
 *
 *   PART D — Test your Function:
 *
 *     AWS Console → Lambda → your function → "Test" tab
 *       → Click "Create new event"
 *       → Event name     : test-event
 *       → Event JSON (paste this):
 *           {
 *             "queryStringParameters": {
 *               "name": "Hitesh"
 *             }
 *           }
 *       → Click "Save"
 *       → Click "Test"
 *
 *     Expected response:
 *     ┌──────────────────────────────────────┐
 *     │  Status: 200                         │
 *     │  Body: { "message": "Hello, Hitesh" }│
 *     │  Duration: 3.45 ms                   │
 *     │  Memory used: 67 MB                  │
 *     └──────────────────────────────────────┘
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 6 — Triggers (What Invokes Lambda)
 * ───────────────────────────────────────────────────────
 *
 *   A Lambda function does nothing on its own — it needs a trigger.
 *   Here are the most common ones with real-world examples.
 *
 *
 *   Trigger 1 — API Gateway (most common):
 *   - Expose your Lambda as an HTTP endpoint.
 *   - User hits a URL → API Gateway invokes Lambda → Lambda
 *     processes the request → returns an HTTP response.
 *   - This is how you build serverless APIs and backends.
 *
 *    [Browser / Mobile App]
 *         │
 *         │  GET https://api.chaicode.com/users
 *         ▼
 *    [API Gateway]
 *    Receives HTTP request, maps it to Lambda
 *         │
 *         ▼
 *    [Lambda Function: getUsers]
 *    Queries database, formats response
 *         │
 *         ▼
 *    [API Gateway]
 *    Returns HTTP response to client
 *         │
 *         ▼
 *    [Browser / Mobile App]
 *    Receives JSON response ✓
 *
 *
 *   Trigger 2 — S3 (file processing):
 *   - Automatically process files as soon as they are uploaded.
 *   - e.g., user uploads a profile photo → Lambda resizes it
 *     and saves thumbnails → no manual intervention needed.
 *
 *    [User uploads image]
 *         │
 *         ▼
 *    [S3 Bucket: uploads]
 *    File saved, S3 fires event
 *         │
 *         ▼
 *    [Lambda Function: resizeImage]
 *    Reads original image
 *    Creates thumbnail (200x200)
 *    Saves to S3 Bucket: thumbnails
 *         │
 *         ▼
 *    Thumbnail available in seconds ✓
 *    No server running — paid only for the 300ms Lambda ran
 *
 *
 *   Trigger 3 — CloudWatch Events / EventBridge (scheduled jobs):
 *   - Run Lambda on a schedule — like a cron job but serverless.
 *   - No EC2 instance needed just to run a nightly report.
 *
 *    Schedule: every day at 2 AM
 *         │
 *         ▼
 *    [CloudWatch Events / EventBridge]
 *    Fires at 2 AM
 *         │
 *         ▼
 *    [Lambda Function: dailyReport]
 *    Queries database
 *    Generates PDF report
 *    Sends to S3
 *    Emails stakeholders via SES
 *         │
 *         ▼
 *    Done — no server running all day ✓
 *
 *
 *   Trigger 4 — SQS (queue processing):
 *   - Lambda polls an SQS queue and processes messages in batches.
 *   - Great for background jobs and decoupled architectures.
 *
 *    [Order placed on website]
 *         │
 *         ▼
 *    [SQS Queue: order-processing]
 *    Message added to queue
 *         │
 *         ▼
 *    [Lambda Function: processOrder]
 *    Triggered when messages arrive
 *    Updates inventory
 *    Charges payment
 *    Sends confirmation email
 *         │
 *         ▼
 *    Order processed asynchronously ✓
 *
 *
 *   Trigger 5 — DynamoDB Streams:
 *   - React to database changes in real time.
 *   - Every insert, update, or delete in DynamoDB fires a Lambda.
 *
 *    [New user record inserted in DynamoDB]
 *         │
 *         ▼
 *    [DynamoDB Stream]
 *    Captures the change event
 *         │
 *         ▼
 *    [Lambda Function: onNewUser]
 *    Sends welcome email via SES
 *    Creates user profile in another table
 *    Sends Slack notification to team
 *         │
 *         ▼
 *    All side effects handled automatically ✓
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 7 — Lambda + API Gateway (Serverless API)
 * ───────────────────────────────────────────────────────
 *
 *   This is the most common Lambda pattern — building a full
 *   REST API without managing any servers.
 *
 *
 *   How to connect API Gateway to Lambda:
 *
 *   PART A — Create the API:
 *
 *     AWS Console
 *       → Search "API Gateway"
 *       → Click "Create API"
 *       → Choose: "HTTP API" (simpler) or "REST API" (more features)
 *       → API name  : chaicode-api
 *       → Click "Next" → "Next" → "Create"
 *
 *   PART B — Add a Route:
 *
 *     API Gateway Console → chaicode-api
 *       → Click "Routes" → "Create"
 *       → Method : GET
 *       → Path   : /users
 *       → Integration: Lambda Function → select your function
 *       → Click "Create"
 *
 *   PART C — Deploy the API:
 *
 *     API Gateway Console → chaicode-api
 *       → Click "Deploy"
 *       → Stage: $default (or create a named stage like "prod")
 *       → Your API URL is now live:
 *         https://<api-id>.execute-api.ap-south-1.amazonaws.com/users
 *
 *
 *   Full serverless API routing:
 *
 *    API Gateway endpoint: https://api.chaicode.com
 *
 *    GET  /users        → Lambda: getUsers
 *    POST /users        → Lambda: createUser
 *    GET  /users/{id}   → Lambda: getUserById
 *    PUT  /users/{id}   → Lambda: updateUser
 *    DELETE /users/{id} → Lambda: deleteUser
 *
 *    Each route → its own Lambda function
 *    Each function scales independently ✓
 *    No server running when nobody calls the API ✓
 *
 *
 *   VISUAL — Serverless API architecture:
 *
 *    ┌──────────────────────────────────────────────────────┐
 *    │                  [Client / Browser]                  │
 *    └──────────────────────────┬───────────────────────────┘
 *                               │  HTTPS request
 *                               ▼
 *    ┌──────────────────────────────────────────────────────┐
 *    │  API Gateway                                         │
 *    │  Routes requests to the right Lambda function        │
 *    └───┬──────────┬──────────┬──────────┬─────────────────┘
 *        │          │          │          │
 *        ▼          ▼          ▼          ▼
 *    [Lambda]   [Lambda]   [Lambda]   [Lambda]
 *    getUsers   createUser getUserById deleteUser
 *        │          │          │          │
 *        └──────────┴──────────┴──────────┘
 *                         │
 *                         ▼
 *                   [Database]
 *                   DynamoDB / RDS
 *
 *
 * ───────────────────────────────────────────────────────────
 * STEP 8 — Lambda Execution Model (How Lambda Actually Runs)
 * ───────────────────────────────────────────────────────────
 *
 *   Understanding how Lambda executes internally helps you write
 *   better Lambda functions.
 *
 *
 *   Lambda execution lifecycle:
 *
 *    ┌──────────────────────────────────────────────────────┐
 *    │  Phase 1: INIT (runs once per cold start)            │
 *    │                                                      │
 *    │  - AWS downloads your code from S3                   │
 *    │  - AWS starts the runtime (Node.js, Python etc.)     │
 *    │  - Your code OUTSIDE the handler runs here           │
 *    │    e.g., const db = connectToDatabase();             │
 *    │         This runs once and is reused ✓               │
 *    └──────────────────────────────────────────────────────┘
 *                          │
 *                          ▼
 *    ┌──────────────────────────────────────────────────────┐
 *    │  Phase 2: INVOKE (runs on every request)             │
 *    │                                                      │
 *    │  - AWS calls your handler function                   │
 *    │  - Your handler runs with the event data             │
 *    │  - Returns a response                                │
 *    │  - This phase repeats for each request               │
 *    └──────────────────────────────────────────────────────┘
 *                          │
 *                          ▼
 *    ┌──────────────────────────────────────────────────────┐
 *    │  Phase 3: SHUTDOWN (after idle timeout)              │
 *    │                                                      │
 *    │  - AWS recycles the execution environment            │
 *    │  - Any data stored outside the handler is lost       │
 *    │  - Next request starts a new cold start              │
 *    └──────────────────────────────────────────────────────┘
 *
 *
 *   Practical implication — initialise outside the handler:
 *
 *    BAD (connects to DB on every request — slow and expensive):
 *
 *    export const handler = async (event) => {
 *      const db = await connectToDatabase(); // runs every time ✗
 *      const users = await db.query('SELECT * FROM users');
 *      return users;
 *    };
 *
 *    GOOD (connects to DB once, reuses on warm starts):
 *
 *    const db = await connectToDatabase(); // runs once on INIT ✓
 *
 *    export const handler = async (event) => {
 *      const users = await db.query('SELECT * FROM users'); // reuses
 *      return users;
 *    };
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 9 — Lambda Versions and Aliases
 * ───────────────────────────────────────────────────────
 *
 *   Versions:
 *   - Every time you publish your Lambda code, AWS creates
 *     a numbered Version (v1, v2, v3...).
 *   - Versions are immutable — once published, the code cannot change.
 *   - $LATEST always points to the most recent unpublished code.
 *   - This allows you to roll back to any previous version.
 *
 *    Version history:
 *    ┌──────────────┐
 *    │  $LATEST     │ ← always your current working code
 *    ├──────────────┤
 *    │  Version 3   │ ← published, immutable
 *    ├──────────────┤
 *    │  Version 2   │ ← published, immutable
 *    ├──────────────┤
 *    │  Version 1   │ ← published, immutable
 *    └──────────────┘
 *
 *
 *   Aliases:
 *   - An Alias is a pointer to a specific version.
 *   - Instead of updating all your API Gateway configs every deploy,
 *     you just update the alias to point to the new version.
 *   - Common aliases: prod, staging, dev
 *
 *    ┌──────────────────────────────────────────────┐
 *    │  Alias: prod    → points to Version 3        │
 *    │  Alias: staging → points to Version 4        │
 *    │  Alias: dev     → points to $LATEST          │
 *    └──────────────────────────────────────────────┘
 *
 *   Traffic splitting with aliases:
 *   - You can split traffic between two versions.
 *   - e.g., send 90% of traffic to v3 and 10% to v4 (canary deploy).
 *   - If v4 has issues, only 10% of users are affected.
 *   - Roll back by updating the alias split to 100% v3.
 *
 *    prod alias traffic split:
 *    90% → Version 3 (stable)
 *    10% → Version 4 (testing in production)
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 10 — Lambda Monitoring and Logs
 * ───────────────────────────────────────────────────────
 *
 *   - Every Lambda invocation automatically sends logs to
 *     AWS CloudWatch Logs — no setup needed.
 *   - Every console.log() in your function appears in CloudWatch.
 *   - Lambda also automatically reports key metrics to CloudWatch:
 *
 *     +---------------------------+----------------------------------+
 *     |  Metric                   |  What it tells you               |
 *     +---------------------------+----------------------------------+
 *     |  Invocations              |  How many times function ran     |
 *     |  Duration                 |  How long each run took          |
 *     |  Errors                   |  How many invocations failed     |
 *     |  Throttles                |  How many were rejected (limit)  |
 *     |  ConcurrentExecutions     |  How many ran simultaneously     |
 *     |  ColdStarts               |  How many were cold starts       |
 *     +---------------------------+----------------------------------+
 *
 *
 *   How to view Lambda logs:
 *
 *     Option A — AWS Console:
 *     Lambda Console → your function → "Monitor" tab
 *       → "View logs in CloudWatch" → see all log streams
 *
 *     Option B — AWS CLI:
 *     $ aws logs tail /aws/lambda/hello-world --follow
 *
 *
 *   How to set up an alert for Lambda errors:
 *
 *     CloudWatch Console
 *       → Alarms → "Create Alarm"
 *       → Metric: Lambda → hello-world → Errors
 *       → Condition: Errors > 0
 *       → Action: Send SNS notification → email you
 *       → Alarm name: lambda-hello-world-errors
 *
 *
 * ───────────────────────────────────────────────────────
 * STEP 11 — Delete Lambda Resources (Clean Up)
 * ───────────────────────────────────────────────────────
 *
 *   Lambda itself has no ongoing cost when not running.
 *   But clean up API Gateway and related resources:
 *
 *   PART A — Delete the Lambda Function:
 *
 *     Lambda Console
 *       → Functions → select your function
 *       → Click "Actions" → "Delete"
 *       → Type "delete" to confirm → Click "Delete"
 *
 *     What happens:
 *     - All versions and aliases are deleted.
 *     - CloudWatch log group remains (delete separately if needed).
 *     - IAM execution role remains (delete separately).
 *
 *   PART B — Delete API Gateway (if created):
 *
 *     API Gateway Console
 *       → Select your API → "Actions" → "Delete"
 *       → Confirm → Click "Delete"
 *
 *   PART C — Delete CloudWatch Log Group (optional):
 *
 *     CloudWatch Console
 *       → Log groups → /aws/lambda/hello-world
 *       → "Actions" → "Delete log group"
 *
 *   PART D — Delete IAM Role (optional):
 *
 *     IAM Console
 *       → Roles → search for your Lambda role
 *       → Select → "Delete"
 *
 *   Clean up order:
 *   1. Delete Lambda Function
 *   2. Delete API Gateway (if created)
 *   3. Delete CloudWatch Log Group (optional)
 *   4. Delete IAM Role (optional)
 *
 *
 * ───────────────────────────────────────────────────────
 * FINAL ARCHITECTURE — Lambda + API Gateway + DynamoDB
 * ───────────────────────────────────────────────────────
 *
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │                    INTERNET                                 │
 *    └────────────────────────┬────────────────────────────────────┘
 *                             │  User calls API
 *                             ▼
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  API Gateway                                                │
 *    │  https://api.chaicode.com                                   │
 *    │  Routes by method + path                                    │
 *    └──────┬───────────┬───────────┬───────────┬──────────────────┘
 *           │           │           │           │
 *    GET /users  POST /users  GET /users/{id}  DELETE /users/{id}
 *           │           │           │           │
 *           ▼           ▼           ▼           ▼
 *       [Lambda]    [Lambda]    [Lambda]    [Lambda]
 *       getUsers   createUser  getUserById deleteUser
 *       ~50ms       ~80ms       ~50ms        ~60ms
 *           │           │           │           │
 *           └───────────┴───────────┴───────────┘
 *                               │
 *                               ▼
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  DynamoDB (or RDS)                                          │
 *    │  Database — only accessible from Lambda via VPC             │
 *    └─────────────────────────────────────────────────────────────┘
 *
 *    Supporting Services:
 *    ┌─────────────────────────────────────────────────────────────┐
 *    │  CloudWatch  → automatic logs for every Lambda invocation   │
 *    │  IAM Role    → gives Lambda permission to access DynamoDB   │
 *    │  EventBridge → scheduled Lambda triggers (cron jobs)        │
 *    │  SQS         → async Lambda triggers (queue processing)     │
 *    │  S3          → trigger Lambda on file upload events         │
 *    └─────────────────────────────────────────────────────────────┘
 *
 *
 *   When to use EC2 vs ECS vs Lambda:
 *
 *     +----------------------------+-------+-------+---------+
 *     |  Situation                 |  EC2  |  ECS  |  Lambda |
 *     +----------------------------+-------+-------+---------+
 *     |  Long running web server   |  ✓    |  ✓    |         |
 *     |  Containerised app         |       |  ✓    |         |
 *     |  Need full OS access       |  ✓    |       |         |
 *     |  GPU / ML training         |  ✓    |       |         |
 *     |  Microservices             |       |  ✓    |  ✓      |
 *     |  REST API (serverless)     |       |       |  ✓      |
 *     |  Event-driven tasks        |       |       |  ✓      |
 *     |  Scheduled cron jobs       |       |       |  ✓      |
 *     |  File processing           |       |       |  ✓      |
 *     |  Unpredictable traffic     |       |       |  ✓      |
 *     |  Zero idle cost needed     |       |       |  ✓      |
 *     |  Task < 15 minutes         |       |       |  ✓      |
 *     +----------------------------+-------+-------+---------+
 *
 *
 *   Summary — what each component does:
 *
 *     +------------------+------------------------------------------+
 *     |  Component       |  Responsibility                          |
 *     +------------------+------------------------------------------+
 *     |  Lambda Function |  Your actual code that runs on demand    |
 *     +------------------+------------------------------------------+
 *     |  Trigger         |  What invokes the function (API, S3 etc) |
 *     +------------------+------------------------------------------+
 *     |  Event           |  Data passed to the function on invoke   |
 *     +------------------+------------------------------------------+
 *     |  Handler         |  Entry point AWS calls in your code      |
 *     +------------------+------------------------------------------+
 *     |  Execution Role  |  IAM role giving Lambda AWS permissions  |
 *     +------------------+------------------------------------------+
 *     |  Layer           |  Shared libraries across functions       |
 *     +------------------+------------------------------------------+
 *     |  Version         |  Immutable published snapshot of code    |
 *     +------------------+------------------------------------------+
 *     |  Alias           |  Named pointer to a version (prod, dev)  |
 *     +------------------+------------------------------------------+
 *     |  API Gateway     |  HTTP front door that triggers Lambda    |
 *     +------------------+------------------------------------------+
 *     |  CloudWatch Logs |  Automatic log collection for every run  |
 *     +------------------+------------------------------------------+
 */

In [ ]:
/**
 * AWS Services — Why So Many?
 *
 * - At its core, AWS started as a cloud provider whose primary job was
 *   to expose the physical layer through a hypervisor — essentially
 *   renting out raw virtual machines to users.
 *
 * - But AWS realized: "We already have the infrastructure and the
 *   engineers — why not build common, reusable services on top of it
 *   and offer them to developers as managed services?"
 *
 * - Here is how that thinking evolved:
 *
 *   The Origin Story (URL Shortener Example):
 *   - Suppose you know how to code, so you build a URL Shortener app
 *     and deploy it on your own server (raw machine from AWS).
 *   - You then offer it to users: "You don't need to build your own
 *     URL Shortener — just use mine. I'll charge $1 per 1000 URLs."
 *   - This is exactly what AWS did at a much larger scale:
 *     - AWS already had the infrastructure.
 *     - AWS hired developers and built commonly needed services.
 *     - Instead of every user building the same thing from scratch,
 *       AWS packaged it and offered it as a managed service.
 *
 *   Two options AWS gives every user:
 *
 *     +---------------------------------------------------------------+
 *     |                        [User]                                 |
 *     |                           |                                   |
 *     |          +----------------+----------------+                  |
 *     |          |                                 |                  |
 *     |          V                                 V                  |
 *     |  +----------------+             +--------------------+        |
 *     |  |  Raw Machine   |             |  Managed Service   |        |
 *     |  |  (EC2 / VM)    |             |  (SQS, SES, ECS..) |        |
 *     |  |                |             |                    |        |
 *     |  | User installs  |             | AWS manages the    |        |
 *     |  | and manages    |             | infra, scaling,    |        |
 *     |  | everything     |             | patching, uptime   |        |
 *     |  | themselves.    |             | for you.           |        |
 *     |  +----------------+             +--------------------+        |
 *     |          |                                 |                  |
 *     |          +----------------+----------------+                  |
 *     |                           |                                   |
 *     |                           V                                   |
 *     |              +------------------------+                       |
 *     |              |  AWS Physical Layer /  |                       |
 *     |              |     Hypervisor         |                       |
 *     |              +------------------------+                       |
 *     +---------------------------------------------------------------+
 *
 *   A Better Example — SQS (Simple Queue Service):
 *   - AWS noticed that almost every full-stack developer building on
 *     their platform was implementing a message queue from scratch.
 *   - Instead of every user reinventing the wheel, AWS built SQS
 *     (Simple Queue Service) — a fully managed queue — and offered
 *     it as a service.
 *   - The user no longer needs to:
 *     - Spin up a raw machine
 *     - Install and configure a queue (like RabbitMQ or Kafka)
 *     - Handle scaling, failures, or message persistence themselves
 *   - They simply use SQS and pay for what they consume.
 *
 *   How AWS thinks about building a new service:
 *
 *     +---------------------------------------------------------------+
 *     |                                                               |
 *     |   "Are many of our customers building X from scratch          |
 *     |    on top of raw EC2 machines?"                               |
 *     |                           |                                   |
 *     |              +------------+------------+                      |
 *     |              |                         |                      |
 *     |             YES                        NO                     |
 *     |              |                         |                      |
 *     |              V                         V                      |
 *     |   Build it as a managed         Leave it to the user.         |
 *     |   service and offer it          Raw EC2 is enough.            |
 *     |   on AWS.                                                     |
 *     +---------------------------------------------------------------+
 *
 *   Examples of AWS Managed Services built this way:
 *
 *     +---------------------------+------------------------------------+
 *     |  Common Need              |  AWS Managed Service               |
 *     +---------------------------+------------------------------------+
 *     |  a. Send Emails           |  SES  (Simple Email Service)       |
 *     |  b. Manage Containers     |  ECS  (Elastic Container Service)  |
 *     |  c. Message Queue         |  SQS  (Simple Queue Service)       |
 *     |  d. Video Transcoding     |  AWS Elastic Transcoder /          |
 *     |                           |  AWS MediaConvert                  |
 *     |  e. Store Files/Objects   |  S3   (Simple Storage Service)     |
 *     |  f. Managed Database      |  RDS, DynamoDB, Aurora             |
 *     |  g. Run Code (Serverless) |  AWS Lambda                        |
 *     |  h. DNS Management        |  Route 53                          |
 *     |  i. Monitoring & Alerts   |  CloudWatch                        |
 *     +---------------------------+------------------------------------+
 *
 *   - This is exactly why the AWS ecosystem is so large — almost every
 *     common infrastructure problem that developers repeatedly solve
 *     has been packaged by AWS into a managed service.
 *   - The result: developers focus on writing business logic, and AWS
 *     handles the heavy lifting of infrastructure, scaling, availability,
 *     and maintenance.
 *
 *   Layers of AWS (summary):
 *
 *     +---------------------------------------------------------------+
 *     |  Layer 3: Managed Services                                    |
 *     |  (SQS, SES, ECS, RDS, S3, Lambda, CloudWatch, Route53...)     |
 *     +---------------------------------------------------------------+
 *     |  Layer 2: Virtualization / APIs                               |
 *     |  (AWS APIs, CLI, SDKs, Management Console)                    |
 *     +---------------------------------------------------------------+
 *     |  Layer 1: Physical Layer                                      |
 *     |  (Servers, Storage, Network Cards, Cooling, Power)            |
 *     +---------------------------------------------------------------+
 */

In [ ]:
/**
 * AWS Video Platform LMS:
 * - Red Button: POST /end-class/:classID
 *   - DB status update class_status = 'Ended'
 *   - Raise an event "class_ended" - Event Bridge
 * - Jab class chl rhi thi to kuch system hoga to class ko record kr
 *   rha hoga, aur ye recording system mere recording to S3 m daal rha
 *   hoga.
 * - Agar humari class 2.5 hrs ki hui to kya ye 2.5hrs baad daalenga
 *   using the recording infrastructure like AWS Chime? No! Kyuki agar 
 *   chalte chlte 2.4hrs pe class crash kr gyi to meri saare recording 
 *   gyi, isliye humein periodically usse S3 m daal deni chaiye.
 * - Ab iske andar jab hum recording enable krte hai to ye har 5 sec
 *   k baad chunks ko S3 m daalte the, aur jab user ne end ka button
 *   dabaya to humari call POST /end-class/:classID pe aai, aur humne
 *   ek event raise kiya class ended ka jiske liye hum use kr rhe the
 *   'EventBridge'. Using this service, hum ek event raise kr skte hai
 *   and then we can trigger something based on that. It is a serverless
 *   fully managed event bus service that simplifies connecting apps
 *   using data from your own apps. Humein ek event bridge lagaya aur
 *   iss event bridge k basis pe ek lambda attach kr diya, aur jab v
 *   ye class end hogi to ek event trigger hogi jo uss class k liye
 *   ek serverless function invoke hoga jisse video compilation start
 *   jo jaega kyuki S3 m bahut saare chunks pade hai, So lambda k andr
 *   code hoga ki read the folder from S3 and then merge all the files.
 * - But 2.5hrs k liye hum lambda invoke kr k nhi rknge. So how to
 *   merge these chunk videos in AWS? Lambda function /meeting/id ka
 *   path dete the jaha pe saare chunks pade hai uska path hum dete the
 *   to this AWS Elastic Transcoder jiske hum kehte the ki S3 se saare
 *   chunks uthao aur:
 *   - Chunks ko stitch karo per job
 *   - After that uske complete hone pe AWS Elastic Trnascoder saare
 *     video files ko S3 pe rakhega jisme humne ek path de rkha hai
 *     /meeting/id/stitched. And S3 pe upar hum event bridge laga skte
 *     hai ki agar iss path k upar koi v file write hoti(PUT) hai to 
 *     humein ek event bridge de dena (/meeting/**). And iske basis
 *     pe hum ek aur lambda trigger kr skte hai and iss lambda ko pta
 *     hai ki mera video ready ho gya hai, aur ye db ko update krta
 *     tha from 'processing' to 'processed'. 
 * - Ab ek video file 3-4GB k banti hai, aur jo humara client hai usne
 *   video request kiya to kya 3-4 GB k video usse load karwaenge?
 *   No! Iss User Experience kharab hoga aur 'Egress' cost v high hoga
 *   (jo data bahar jaa rha hai).
 * - Ab jitan data bahar jaata hai uske base pe AWS charge krti hai
 *   aur iss approach se humara cost high hoga.
 * - To ab hum video k multiple formats banaenge:
 *   - 480p
 *   - 720p
 *   - 1080p
 *   - 4k
 *     +
 *   - Chunks v bananege so that hum user ko poori video ek saath na de
 * - So, agar use ka internet slow hai to humein 480p stream krna chahiye
 *   but agar internet fast hai to 4k pe stream chunks behjnge. So,
 *   abhi 'processing' m hi database rahega, aur video ek baar jab stitch
 *   ho gyi to hum dobara se AWS media convert use krnge aur isko hum
 *   original video de denge, and we will ask it to output:
 *   - 480p
 *   - 720p
 *   - 1080p
 *   - 4k
 * - Aur ye S3 pe jaa k inn videos ko store karega. Then S3 humein dobara
 *   ek event dega jo humein multiple times aaega for every format because
 *   ye videos alag alag time pe process hogi, and because 360 sbse
 *   chota hota hai to sbse pehle iska aaega.
 * - To iss event bridge k baad hum ek aur lambda invoke krte the jo
 *   db m jaa k ek status update krta tha 'completedFormats'. Aur
 *   jo format aaya bagal m usse rkh denge:
 *   - completedFormats 360
 *   - completedFormats 480
 *   ....
 *   ....
 * - Aur jab saare ho jaenge to hum apna status 'processed' mark kr denge.
 * - Aur ab jb user frontend pe aata hai aur player load krta hai to
 *   mere paas saare format hai to hum usse efficient streaming de skte
 *   hai.
 * 
 * Why this system failed?
 * - Too much lambda used, which causes
*/